Figure out functions to process a user dataset and import it to the SQLite schema.

In [1]:
import re
import pandas as pd
import numpy as np
import sqlite3
from contextlib import contextmanager
from pathlib import Path
import uuid
import datetime
import csv
import tomllib

In [2]:
#Helpers

DB_FILE = Path('../sqlite_backend.db')


@contextmanager
def get_db_connection():
    conn = sqlite3.connect(DB_FILE)
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

Assume we receive this request:

In [3]:
request = {
    'parameters': {
        'uuid': None,#'0123456789ABCDEF0123456789ABCDEF', #Manually inserted as test user
        'datasetName':'MyTestDecryptMDataset',
        'datasetType': 'Curve',#Curve/FoldChange
        'omics': 'Phosphorylation',#Phosphorylation/Other/Protein
        'hasFoldChangeColumn': 1, #1/0
        'foldChangeDataFoldChangeScale': 'raw', #raw/log/none
        'taxcode': 9606, #9606/10090
        
    },
    'file': {
        #TODO: See if this is possible or if you can only do a list
        'csvFile': 'C:\\Users\\jmueller\\Downloads\\examples\\decryptm_data\\CurveCurator\\curves.txt',
        #'csvFile': '/home/jmueller/Downloads/examples/decryptm_data/CurveCurator/curves.txt',
        #'csvFile': '/home/jmueller/Downloads/examples/fold_change_data/volcano_ptm_data.csv',
        'tomlFile': 'C:\\Users\\jmueller\\Downloads\\examples\\decryptm_data\\CurveCurator\\parameters.toml'
        #'tomlFile': '/home/jmueller/Downloads/examples/decryptm_data/CurveCurator/parameters.toml' 
    }
}
request

{'parameters': {'uuid': None,
  'datasetName': 'MyTestDecryptMDataset',
  'datasetType': 'Curve',
  'omics': 'Phosphorylation',
  'hasFoldChangeColumn': 1,
  'foldChangeDataFoldChangeScale': 'raw',
  'taxcode': 9606},
 'file': {'csvFile': 'C:\\Users\\jmueller\\Downloads\\examples\\decryptm_data\\CurveCurator\\curves.txt',
  'tomlFile': 'C:\\Users\\jmueller\\Downloads\\examples\\decryptm_data\\CurveCurator\\parameters.toml'}}

We start with obtaining a user id, optionally by creating it if it does not exist yet.

In [4]:
def getUserIdFromUUID(input_uuid):
    if not input_uuid:
        return None
    with get_db_connection() as conn:
        res = conn.execute('SELECT U.USER_ID FROM USER U WHERE U.SESSION_ID = ?', [input_uuid]).fetchall()
        if len(res) > 0:
            return res[0][0]
        else:
            return None

In [5]:
def getOrCreateUserId(input_uuid):
    user_id = getUserIdFromUUID(input_uuid)
    if not user_id:
        #Create new and insert
        new_uuid = str(uuid.uuid4()).replace('-', '').upper()
        with get_db_connection() as conn:
            conn.execute('INSERT INTO USER(SESSION_ID,LAST_ACCESSION_DATE) VALUES (?,?)', [new_uuid, datetime.datetime.now().isoformat()])
        user_id = getUserIdFromUUID(new_uuid)
    return user_id

In [6]:
user_id = getOrCreateUserId(request['parameters']['uuid'])
user_id

8

Now we create an entry into the user_dataset table:

In [7]:
def insertToUserDataset(dataset_name, user_id, dataset_type, omics, taxcode):
    with get_db_connection() as conn:
        conn.execute('INSERT INTO USER_DATASET(NAME, USER_ID, DATASET_TYPE, OMICS, TAXCODE) VALUES (?,?,?,?,?)',
                     [dataset_name, user_id, dataset_type, omics, taxcode])
        #Maybe necessary later: Return the id of the new dataset
        dataset_id = conn.execute('SELECT UD.DATASET_ID FROM USER_DATASET UD WHERE UD.USER_ID = ? AND UD.NAME = ?', 
                                  [user_id, dataset_name]
                                 ).fetchall()[0][0]
        return dataset_id

In [8]:
dataset_id = insertToUserDataset(
    request['parameters']['datasetName'],
    user_id,
    request['parameters']['datasetType'],
    request['parameters']['omics'],
    request['parameters']['taxcode'],
)
dataset_id

10

We assume we have that figured out and just parse the file into pandas

In [9]:
def get_delimiter(file_path, bytes=50000):
    sniffer = csv.Sniffer()
    data = open(file_path, "r").read(bytes)
    delimiter = sniffer.sniff(data).delimiter
    return delimiter

In [10]:
csv_delimiter = get_delimiter(request['file']['csvFile'], 50000)

In [11]:
input_csv_df = pd.read_csv(request['file']['csvFile'], sep=csv_delimiter)
input_csv_df

,Modified sequence,N duplicates,Genes,Proteins,Score,Raw 1,Raw 2,Raw 3,Raw 4,Raw 5,...,Curve Back Error,Null Model,Null RMSE,Curve F_Value,Curve P_Value,Curve Log P_Value,Curve F_Value SAM Corrected,Curve Relevance Score,Curve Regulation,Curve q_Value
0,(ac)AAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,182.81,104000.0,108540.0,108270.0,129490.0,137980.0,...,4.779014e-02,1.014290,0.045691,1.474121,0.347278,0.459323,0.145907,0.000206,not,0.629313
1,(ac)AAAAAAAGDS(ph)DSWDADAFSVEDPVRK,6,EIF3J,O75822;O75822-2;O75822-3,233.50,325843.5,355756.6,337716.3,372103.7,376749.5,...,1.306756e-02,0.978271,0.026881,6.393457,0.017295,1.762080,0.088781,-0.000000,not,0.719848
2,(ac)AAAAAAAGDSDS(ph)WDADAFSVEDPVRK,3,EIF3J,O75822;O75822-2;O75822-3,170.56,546448.0,621476.0,573421.0,646003.0,671111.0,...,1.673664e-02,1.017453,0.025695,0.645374,0.751752,0.123925,0.015688,-0.000000,not,0.940467
3,(ac)AAAAAAAGDSDSWDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,116.35,42125.0,44422.0,41461.0,48380.0,47313.0,...,2.127932e-02,0.998084,0.047544,1.564582,0.320002,0.494848,0.082689,-0.000000,not,0.736682
4,(ac)AAAAPDSRVS(ph)EEENLKK,1,RRP15,Q9Y3B9,168.05,803650.0,889630.0,877020.0,925750.0,993160.0,...,2.484183e-02,1.050177,0.054634,1.329570,0.396697,0.401541,0.089889,-0.000000,not,0.717919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19604,YYGGGSEGGR,1,HNRNPL,P14866,177.78,108820.0,109800.0,104720.0,122130.0,121740.0,...,1.799273e+09,0.950525,0.051942,2.212674,0.184866,0.733144,0.299523,0.017102,not,0.500095
19605,YYLHDDR,2,THRAP3,Q9Y2W1;A0A3B3ITZ9,158.36,452249.0,457444.0,425540.0,457611.0,497938.0,...,1.248265e-02,0.908433,0.039181,3.940836,0.057164,1.242875,0.034530,-0.000000,not,0.873684
19606,YYRSPS(ph)R,1,RSRP1,H3BMX3;H3BTJ0;Q9BUV0-3;Q9BUV0-2;Q9BUV0,76.10,101500.0,117450.0,103280.0,105530.0,111450.0,...,2.143058e-02,0.978881,0.051361,0.795917,0.656135,0.183007,0.078824,-0.000000,not,0.745592
19607,YYS(ph)DS(ph)DDELTVEQR,1,BOD1L1,Q8NFC6,255.16,310360.0,366690.0,326720.0,360350.0,394830.0,...,1.889797e-02,1.003857,0.043479,2.968655,0.105661,0.976087,0.108746,-0.000000,not,0.680164


In [12]:
expected_colnames_fcdata_map = {
    'Modified sequence': ['modifiedsequence','modsequence'],
    'Psite': ['psite', 'p-site'],
    'Gene Names': ['gene_names', 'genes', 'genenames'],
    'Uniprot': ['proteinids', 'proteins', 'uniprot', 'uniprot_acc'],
    'Regulation': ['regulation'],
    'Fold Change': ['foldchange','fc','logfc'],
    'Experiment': ['experiment'],
}
additional_expected_colnames_decryptM_map = {
    'pEC50': ['pec50'],
    'pEC50_Error': ['logec50error', 'pec50error'],
    'Slope': ['slope', 'curveslope'],
    'Front': ['front', 'curveslope'],
    'Back': ['back', 'curveslope'],
    'Fold Change': ['curvefoldchange'],
    'Regulation' : ['curveregulation'],
    'Curve q-Value': ['curveq_value', 'curveqvalue'],
    'Relevance Score': ['curverelevancescore', 'relevancescore']
}

In [13]:
if request['parameters']['datasetType'] == 'Curve':
    colnames_variants_lookup = {val: key 
                                for key,vals in list(expected_colnames_fcdata_map.items()) + list(additional_expected_colnames_decryptM_map.items())
                                for val in vals}
else:
    colnames_variants_lookup = {val: key for key,vals in expected_colnames_fcdata_map.items() for val in vals}
colnames_variants_lookup

{'modifiedsequence': 'Modified sequence',
 'modsequence': 'Modified sequence',
 'psite': 'Psite',
 'p-site': 'Psite',
 'gene_names': 'Gene Names',
 'genes': 'Gene Names',
 'genenames': 'Gene Names',
 'proteinids': 'Uniprot',
 'proteins': 'Uniprot',
 'uniprot': 'Uniprot',
 'uniprot_acc': 'Uniprot',
 'regulation': 'Regulation',
 'foldchange': 'Fold Change',
 'fc': 'Fold Change',
 'logfc': 'Fold Change',
 'experiment': 'Experiment',
 'pec50': 'pEC50',
 'logec50error': 'pEC50_Error',
 'pec50error': 'pEC50_Error',
 'slope': 'Slope',
 'curveslope': 'Back',
 'front': 'Front',
 'back': 'Back',
 'curvefoldchange': 'Fold Change',
 'curveregulation': 'Regulation',
 'curveq_value': 'Curve q-Value',
 'curveqvalue': 'Curve q-Value',
 'curverelevancescore': 'Relevance Score',
 'relevancescore': 'Relevance Score'}

TODO: 
-  Check for Expected Columns Names, exit if one is missing 

In [14]:
detail_columns = []
actual_colnames_to_expected_colnames = {}
for col in input_csv_df.columns:
    expected_colname_for_actual_colname = colnames_variants_lookup.get(col.lower().replace(' ', ''))
    if expected_colname_for_actual_colname:
        actual_colnames_to_expected_colnames[col] = expected_colname_for_actual_colname
    else:
        detail_columns.append(col)

In [15]:
input_csv_df.rename(actual_colnames_to_expected_colnames, axis=1, inplace=True)
input_csv_df

,Modified sequence,N duplicates,Gene Names,Uniprot,Score,Raw 1,Raw 2,Raw 3,Raw 4,Raw 5,...,Curve Back Error,Null Model,Null RMSE,Curve F_Value,Curve P_Value,Curve Log P_Value,Curve F_Value SAM Corrected,Relevance Score,Regulation,Curve q-Value
0,(ac)AAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,182.81,104000.0,108540.0,108270.0,129490.0,137980.0,...,4.779014e-02,1.014290,0.045691,1.474121,0.347278,0.459323,0.145907,0.000206,not,0.629313
1,(ac)AAAAAAAGDS(ph)DSWDADAFSVEDPVRK,6,EIF3J,O75822;O75822-2;O75822-3,233.50,325843.5,355756.6,337716.3,372103.7,376749.5,...,1.306756e-02,0.978271,0.026881,6.393457,0.017295,1.762080,0.088781,-0.000000,not,0.719848
2,(ac)AAAAAAAGDSDS(ph)WDADAFSVEDPVRK,3,EIF3J,O75822;O75822-2;O75822-3,170.56,546448.0,621476.0,573421.0,646003.0,671111.0,...,1.673664e-02,1.017453,0.025695,0.645374,0.751752,0.123925,0.015688,-0.000000,not,0.940467
3,(ac)AAAAAAAGDSDSWDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,116.35,42125.0,44422.0,41461.0,48380.0,47313.0,...,2.127932e-02,0.998084,0.047544,1.564582,0.320002,0.494848,0.082689,-0.000000,not,0.736682
4,(ac)AAAAPDSRVS(ph)EEENLKK,1,RRP15,Q9Y3B9,168.05,803650.0,889630.0,877020.0,925750.0,993160.0,...,2.484183e-02,1.050177,0.054634,1.329570,0.396697,0.401541,0.089889,-0.000000,not,0.717919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19604,YYGGGSEGGR,1,HNRNPL,P14866,177.78,108820.0,109800.0,104720.0,122130.0,121740.0,...,1.799273e+09,0.950525,0.051942,2.212674,0.184866,0.733144,0.299523,0.017102,not,0.500095
19605,YYLHDDR,2,THRAP3,Q9Y2W1;A0A3B3ITZ9,158.36,452249.0,457444.0,425540.0,457611.0,497938.0,...,1.248265e-02,0.908433,0.039181,3.940836,0.057164,1.242875,0.034530,-0.000000,not,0.873684
19606,YYRSPS(ph)R,1,RSRP1,H3BMX3;H3BTJ0;Q9BUV0-3;Q9BUV0-2;Q9BUV0,76.10,101500.0,117450.0,103280.0,105530.0,111450.0,...,2.143058e-02,0.978881,0.051361,0.795917,0.656135,0.183007,0.078824,-0.000000,not,0.745592
19607,YYS(ph)DS(ph)DDELTVEQR,1,BOD1L1,Q8NFC6,255.16,310360.0,366690.0,326720.0,360350.0,394830.0,...,1.889797e-02,1.003857,0.043479,2.968655,0.105661,0.976087,0.108746,-0.000000,not,0.680164


TODO: Check if the absolute minimum of essential columns is there

In [16]:
#Blacklist of columns that appear in decryptM/CurveCurator files but should not be imported as Datum Details
curve_columns_not_imported = set({'N duplicates', 'Score', 'Raw 1', 'Raw 2', 'Raw 3', 'Raw 4', 'Raw 5', 'Raw 6', 'Raw 7', 'Raw 8', 'Raw 9', 'Raw 10', 'Raw 11', 
'Name', 'Imputation N', 'Imputation Position', 
'Normalized 1', 'Normalized 2', 'Normalized 3', 'Normalized 4', 'Normalized 5', 'Normalized 6', 'Normalized 7', 'Normalized 8', 'Normalized 9', 'Normalized 10', 'Normalized 11', 
'Ratio 1', 'Ratio 2', 'Ratio 3', 'Ratio 4', 'Ratio 5', 'Ratio 6', 'Ratio 7', 'Ratio 8', 'Ratio 9', 'Ratio 10', 'Ratio 11', 
'Signal Quality', 'R2', 'Curve Front', 'Curve Back', 'Curve AUC', 'Curve RMSE', 'Curve Slope Error', 'Curve Front Error', 'Curve Back Error', 'Null Model', 'Null RMSE', 
'Curve F_Value', 'Curve P_Value', 'Curve Log P_Value', 'Curve F_Value SAM Corrected', 'Curve q-Value'})

In [17]:
if request['parameters']['datasetType'] == 'Curve':
    detail_columns = list(set(detail_columns) - curve_columns_not_imported)
detail_columns

['Curve R2']

Now we get a mapping of modification symbol to modification id

In [18]:
def get_modification_symbol_to_id():
    with get_db_connection() as conn:
        res = conn.execute('SELECT SYMBOL, MODIFICATION_ID FROM MODIFICATION').fetchall()
        return {symbol : modification_id for symbol, modification_id in res}

In [19]:
modification_symbol_to_id = get_modification_symbol_to_id()
modification_symbol_to_id

{'(ph)': 1}

In [20]:
#This can be applied to the regulation column to turn everything into up, down, not or NaN
regulation_variants_map = {
    'up': ['up', 'u', '+'],
    'down': ['down', 'd'],
    'not': ['not', '-'],
    None: [None, 'nan', 'none', 'null']
}
regulation_variants_lookup = {val: key for key,vals in regulation_variants_map.items() for val in vals}
def clean_regulation(raw_regulation_val):
    if pd.isna(raw_regulation_val):
        return None
    res = regulation_variants_lookup.get(raw_regulation_val, -1)
    if res != -1:
        return res
    else:
        #TODO Error
        print(f'Unrecognized regulation category: {raw_regulation_val}')
        return

In [21]:
input_csv_df['Regulation'] = input_csv_df['Regulation'].apply(clean_regulation)
input_csv_df

,Modified sequence,N duplicates,Gene Names,Uniprot,Score,Raw 1,Raw 2,Raw 3,Raw 4,Raw 5,...,Curve Back Error,Null Model,Null RMSE,Curve F_Value,Curve P_Value,Curve Log P_Value,Curve F_Value SAM Corrected,Relevance Score,Regulation,Curve q-Value
0,(ac)AAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,182.81,104000.0,108540.0,108270.0,129490.0,137980.0,...,4.779014e-02,1.014290,0.045691,1.474121,0.347278,0.459323,0.145907,0.000206,not,0.629313
1,(ac)AAAAAAAGDS(ph)DSWDADAFSVEDPVRK,6,EIF3J,O75822;O75822-2;O75822-3,233.50,325843.5,355756.6,337716.3,372103.7,376749.5,...,1.306756e-02,0.978271,0.026881,6.393457,0.017295,1.762080,0.088781,-0.000000,not,0.719848
2,(ac)AAAAAAAGDSDS(ph)WDADAFSVEDPVRK,3,EIF3J,O75822;O75822-2;O75822-3,170.56,546448.0,621476.0,573421.0,646003.0,671111.0,...,1.673664e-02,1.017453,0.025695,0.645374,0.751752,0.123925,0.015688,-0.000000,not,0.940467
3,(ac)AAAAAAAGDSDSWDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,116.35,42125.0,44422.0,41461.0,48380.0,47313.0,...,2.127932e-02,0.998084,0.047544,1.564582,0.320002,0.494848,0.082689,-0.000000,not,0.736682
4,(ac)AAAAPDSRVS(ph)EEENLKK,1,RRP15,Q9Y3B9,168.05,803650.0,889630.0,877020.0,925750.0,993160.0,...,2.484183e-02,1.050177,0.054634,1.329570,0.396697,0.401541,0.089889,-0.000000,not,0.717919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19604,YYGGGSEGGR,1,HNRNPL,P14866,177.78,108820.0,109800.0,104720.0,122130.0,121740.0,...,1.799273e+09,0.950525,0.051942,2.212674,0.184866,0.733144,0.299523,0.017102,not,0.500095
19605,YYLHDDR,2,THRAP3,Q9Y2W1;A0A3B3ITZ9,158.36,452249.0,457444.0,425540.0,457611.0,497938.0,...,1.248265e-02,0.908433,0.039181,3.940836,0.057164,1.242875,0.034530,-0.000000,not,0.873684
19606,YYRSPS(ph)R,1,RSRP1,H3BMX3;H3BTJ0;Q9BUV0-3;Q9BUV0-2;Q9BUV0,76.10,101500.0,117450.0,103280.0,105530.0,111450.0,...,2.143058e-02,0.978881,0.051361,0.795917,0.656135,0.183007,0.078824,-0.000000,not,0.745592
19607,YYS(ph)DS(ph)DDELTVEQR,1,BOD1L1,Q8NFC6,255.16,310360.0,366690.0,326720.0,360350.0,394830.0,...,1.889797e-02,1.003857,0.043479,2.968655,0.105661,0.976087,0.108746,-0.000000,not,0.680164


# I. PTM
## A. Peptide

In [22]:
#Assuming you know it's not decryptM and not full proteome, check whether it is peptide or site:
'Modified sequence' in input_csv_df

True

In [23]:
'Psite' in input_csv_df

False

So this one is peptide level

In [24]:
def clean_sequence(seq):
    """Sometimes a peptides sequence is repeated with ";" or "," in between"""
    return re.split(';|,', seq)[0]

In [25]:
input_csv_df['Modified sequence'] = input_csv_df['Modified sequence'].apply(clean_sequence)
input_csv_df

,Modified sequence,N duplicates,Gene Names,Uniprot,Score,Raw 1,Raw 2,Raw 3,Raw 4,Raw 5,...,Curve Back Error,Null Model,Null RMSE,Curve F_Value,Curve P_Value,Curve Log P_Value,Curve F_Value SAM Corrected,Relevance Score,Regulation,Curve q-Value
0,(ac)AAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,182.81,104000.0,108540.0,108270.0,129490.0,137980.0,...,4.779014e-02,1.014290,0.045691,1.474121,0.347278,0.459323,0.145907,0.000206,not,0.629313
1,(ac)AAAAAAAGDS(ph)DSWDADAFSVEDPVRK,6,EIF3J,O75822;O75822-2;O75822-3,233.50,325843.5,355756.6,337716.3,372103.7,376749.5,...,1.306756e-02,0.978271,0.026881,6.393457,0.017295,1.762080,0.088781,-0.000000,not,0.719848
2,(ac)AAAAAAAGDSDS(ph)WDADAFSVEDPVRK,3,EIF3J,O75822;O75822-2;O75822-3,170.56,546448.0,621476.0,573421.0,646003.0,671111.0,...,1.673664e-02,1.017453,0.025695,0.645374,0.751752,0.123925,0.015688,-0.000000,not,0.940467
3,(ac)AAAAAAAGDSDSWDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,116.35,42125.0,44422.0,41461.0,48380.0,47313.0,...,2.127932e-02,0.998084,0.047544,1.564582,0.320002,0.494848,0.082689,-0.000000,not,0.736682
4,(ac)AAAAPDSRVS(ph)EEENLKK,1,RRP15,Q9Y3B9,168.05,803650.0,889630.0,877020.0,925750.0,993160.0,...,2.484183e-02,1.050177,0.054634,1.329570,0.396697,0.401541,0.089889,-0.000000,not,0.717919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19604,YYGGGSEGGR,1,HNRNPL,P14866,177.78,108820.0,109800.0,104720.0,122130.0,121740.0,...,1.799273e+09,0.950525,0.051942,2.212674,0.184866,0.733144,0.299523,0.017102,not,0.500095
19605,YYLHDDR,2,THRAP3,Q9Y2W1;A0A3B3ITZ9,158.36,452249.0,457444.0,425540.0,457611.0,497938.0,...,1.248265e-02,0.908433,0.039181,3.940836,0.057164,1.242875,0.034530,-0.000000,not,0.873684
19606,YYRSPS(ph)R,1,RSRP1,H3BMX3;H3BTJ0;Q9BUV0-3;Q9BUV0-2;Q9BUV0,76.10,101500.0,117450.0,103280.0,105530.0,111450.0,...,2.143058e-02,0.978881,0.051361,0.795917,0.656135,0.183007,0.078824,-0.000000,not,0.745592
19607,YYS(ph)DS(ph)DDELTVEQR,1,BOD1L1,Q8NFC6,255.16,310360.0,366690.0,326720.0,360350.0,394830.0,...,1.889797e-02,1.003857,0.043479,2.968655,0.105661,0.976087,0.108746,-0.000000,not,0.680164


We map protein ids right away bc we won't insert the gene names and uniprots. Peptide ID table is too heavy to retrieve, so we map it from the inside 

In [26]:
with get_db_connection() as conn:
    protein_df = pd.read_sql('SELECT PROTEIN_ID, GENE_NAME, UNIPROT_ACC FROM PROTEIN WHERE TAXCODE = ?',
                             conn,
                             params=[request['parameters']['taxcode']]) 
protein_df

,PROTEIN_ID,GENE_NAME,UNIPROT_ACC
0,1,TRBV18,A0A087X0M5
1,2,TMEM247,A6NEH6
2,3,UNC119B,A6NIH7
3,4,None,A6NJR5
4,5,TMEM278,A6NKF7
...,...,...,...
105714,105715,None,A0A0D9SG52
105715,105716,None,A0A1W2PRQ8
105716,105717,None,C9J4A7
105717,105718,None,G3V3Y1


First merge on Uniprot, then on gene name

In [27]:
#TODO Skip if uniprot is not existent!

In [28]:
uniprot_splits = input_csv_df['Uniprot'].apply(lambda s: re.split(';|,', s)).explode()
uniprot_splits = pd.DataFrame(uniprot_splits)

In [29]:
uniprot_splits = uniprot_splits.reset_index(
).merge(protein_df[['PROTEIN_ID', 'UNIPROT_ACC']],
                   left_on = 'Uniprot',
                   right_on = 'UNIPROT_ACC',
                   how='left'
                  ).dropna(
                  ).drop(['Uniprot', 'UNIPROT_ACC'], axis=1
                        ).rename({'PROTEIN_ID':'PROTEIN_ID_FROM_UNIPROT'},
                                axis=1)
uniprot_splits

,index,PROTEIN_ID_FROM_UNIPROT
0,0,12011.0
1,0,94220.0
2,0,94221.0
3,1,12011.0
4,1,94220.0
...,...,...
75928,19608,96661.0
75929,19608,18448.0
75930,19608,96077.0
75931,19608,50344.0


To resolve ambiguities, group by row index and retain only the smallest protein id (since canonical proteins were inserted first, this should be the most canonical isoform possible)

In [30]:
unique_mapped_proteins_after_uniprot = uniprot_splits.groupby('index').agg('min')
unique_mapped_proteins_after_uniprot

,PROTEIN_ID_FROM_UNIPROT
index,
0,12011.0
1,12011.0
2,12011.0
3,12011.0
4,18595.0
...,...
19604,11525.0
19605,13499.0
19606,6283.0


In [31]:
input_csv_df = input_csv_df.merge(unique_mapped_proteins_after_uniprot,
                   left_index=True,
                   right_index=True,
                   how="left")
input_csv_df

,Modified sequence,N duplicates,Gene Names,Uniprot,Score,Raw 1,Raw 2,Raw 3,Raw 4,Raw 5,...,Null Model,Null RMSE,Curve F_Value,Curve P_Value,Curve Log P_Value,Curve F_Value SAM Corrected,Relevance Score,Regulation,Curve q-Value,PROTEIN_ID_FROM_UNIPROT
0,(ac)AAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,182.81,104000.0,108540.0,108270.0,129490.0,137980.0,...,1.014290,0.045691,1.474121,0.347278,0.459323,0.145907,0.000206,not,0.629313,12011.0
1,(ac)AAAAAAAGDS(ph)DSWDADAFSVEDPVRK,6,EIF3J,O75822;O75822-2;O75822-3,233.50,325843.5,355756.6,337716.3,372103.7,376749.5,...,0.978271,0.026881,6.393457,0.017295,1.762080,0.088781,-0.000000,not,0.719848,12011.0
2,(ac)AAAAAAAGDSDS(ph)WDADAFSVEDPVRK,3,EIF3J,O75822;O75822-2;O75822-3,170.56,546448.0,621476.0,573421.0,646003.0,671111.0,...,1.017453,0.025695,0.645374,0.751752,0.123925,0.015688,-0.000000,not,0.940467,12011.0
3,(ac)AAAAAAAGDSDSWDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,116.35,42125.0,44422.0,41461.0,48380.0,47313.0,...,0.998084,0.047544,1.564582,0.320002,0.494848,0.082689,-0.000000,not,0.736682,12011.0
4,(ac)AAAAPDSRVS(ph)EEENLKK,1,RRP15,Q9Y3B9,168.05,803650.0,889630.0,877020.0,925750.0,993160.0,...,1.050177,0.054634,1.329570,0.396697,0.401541,0.089889,-0.000000,not,0.717919,18595.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19604,YYGGGSEGGR,1,HNRNPL,P14866,177.78,108820.0,109800.0,104720.0,122130.0,121740.0,...,0.950525,0.051942,2.212674,0.184866,0.733144,0.299523,0.017102,not,0.500095,11525.0
19605,YYLHDDR,2,THRAP3,Q9Y2W1;A0A3B3ITZ9,158.36,452249.0,457444.0,425540.0,457611.0,497938.0,...,0.908433,0.039181,3.940836,0.057164,1.242875,0.034530,-0.000000,not,0.873684,13499.0
19606,YYRSPS(ph)R,1,RSRP1,H3BMX3;H3BTJ0;Q9BUV0-3;Q9BUV0-2;Q9BUV0,76.10,101500.0,117450.0,103280.0,105530.0,111450.0,...,0.978881,0.051361,0.795917,0.656135,0.183007,0.078824,-0.000000,not,0.745592,6283.0
19607,YYS(ph)DS(ph)DDELTVEQR,1,BOD1L1,Q8NFC6,255.16,310360.0,366690.0,326720.0,360350.0,394830.0,...,1.003857,0.043479,2.968655,0.105661,0.976087,0.108746,-0.000000,not,0.680164,8770.0


In [32]:
unmapped_proteins_after_uniprot = input_csv_df[pd.isna(input_csv_df['PROTEIN_ID_FROM_UNIPROT'])][['Gene Names']].dropna().drop_duplicates().copy()
unmapped_proteins_after_uniprot['Gene Name'] = unmapped_proteins_after_uniprot['Gene Names'].dropna().apply(lambda s: re.split(';|,', s))
unmapped_proteins_after_uniprot = unmapped_proteins_after_uniprot.explode('Gene Name')
unmapped_proteins_after_uniprot

,Gene Names,Gene Name
1241,SOGA1,SOGA1
1589,BNIP2,BNIP2
14133,BCORL1,BCORL1


In [33]:
mapped_proteins_after_genename = unmapped_proteins_after_uniprot.reset_index(
).merge(protein_df[['PROTEIN_ID', 'GENE_NAME']],
                   left_on = 'Gene Name',
                   right_on = 'GENE_NAME',
                   how='left'
).drop(['Gene Name', 'GENE_NAME', 'Gene Names'], axis=1
).rename({'PROTEIN_ID':'PROTEIN_ID_FROM_GENE_NAME'},
                                axis=1).dropna()
mapped_proteins_after_genename

,index,PROTEIN_ID_FROM_GENE_NAME
1,1589,12041.0
2,1589,23429.0
3,1589,75419.0
4,1589,76124.0
5,14133,5953.0
6,14133,31337.0
7,14133,31338.0
8,14133,62545.0
9,14133,73183.0
10,14133,92470.0


To resolve ambiguities, group by row index and retain only the smallest protein id (since canonical proteins were inserted first, this should be the most canonical isoform possible)

In [34]:
unique_mapped_proteins_after_genename = mapped_proteins_after_genename.groupby('index').agg('min')

Now merge them into the original df and combine with the other protein ids

In [35]:
input_csv_df = input_csv_df.merge(
    unique_mapped_proteins_after_genename,
    left_index=True,
    right_index=True,
    how='left')

In [36]:
def clean_protein_id(row):
    if pd.notna(row['PROTEIN_ID_FROM_UNIPROT']):
        return row['PROTEIN_ID_FROM_UNIPROT']
    elif pd.notna(row['PROTEIN_ID_FROM_GENE_NAME']):
        return row['PROTEIN_ID_FROM_GENE_NAME']
    else:
        return None

In [37]:
input_csv_df['PROTEIN_ID'] = input_csv_df.apply(clean_protein_id, axis=1).astype('Int64')
input_csv_df.drop(['PROTEIN_ID_FROM_UNIPROT', 'PROTEIN_ID_FROM_GENE_NAME'], axis=1, inplace=True)
input_csv_df

,Modified sequence,N duplicates,Gene Names,Uniprot,Score,Raw 1,Raw 2,Raw 3,Raw 4,Raw 5,...,Null Model,Null RMSE,Curve F_Value,Curve P_Value,Curve Log P_Value,Curve F_Value SAM Corrected,Relevance Score,Regulation,Curve q-Value,PROTEIN_ID
0,(ac)AAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,182.81,104000.0,108540.0,108270.0,129490.0,137980.0,...,1.014290,0.045691,1.474121,0.347278,0.459323,0.145907,0.000206,not,0.629313,12011
1,(ac)AAAAAAAGDS(ph)DSWDADAFSVEDPVRK,6,EIF3J,O75822;O75822-2;O75822-3,233.50,325843.5,355756.6,337716.3,372103.7,376749.5,...,0.978271,0.026881,6.393457,0.017295,1.762080,0.088781,-0.000000,not,0.719848,12011
2,(ac)AAAAAAAGDSDS(ph)WDADAFSVEDPVRK,3,EIF3J,O75822;O75822-2;O75822-3,170.56,546448.0,621476.0,573421.0,646003.0,671111.0,...,1.017453,0.025695,0.645374,0.751752,0.123925,0.015688,-0.000000,not,0.940467,12011
3,(ac)AAAAAAAGDSDSWDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,116.35,42125.0,44422.0,41461.0,48380.0,47313.0,...,0.998084,0.047544,1.564582,0.320002,0.494848,0.082689,-0.000000,not,0.736682,12011
4,(ac)AAAAPDSRVS(ph)EEENLKK,1,RRP15,Q9Y3B9,168.05,803650.0,889630.0,877020.0,925750.0,993160.0,...,1.050177,0.054634,1.329570,0.396697,0.401541,0.089889,-0.000000,not,0.717919,18595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19604,YYGGGSEGGR,1,HNRNPL,P14866,177.78,108820.0,109800.0,104720.0,122130.0,121740.0,...,0.950525,0.051942,2.212674,0.184866,0.733144,0.299523,0.017102,not,0.500095,11525
19605,YYLHDDR,2,THRAP3,Q9Y2W1;A0A3B3ITZ9,158.36,452249.0,457444.0,425540.0,457611.0,497938.0,...,0.908433,0.039181,3.940836,0.057164,1.242875,0.034530,-0.000000,not,0.873684,13499
19606,YYRSPS(ph)R,1,RSRP1,H3BMX3;H3BTJ0;Q9BUV0-3;Q9BUV0-2;Q9BUV0,76.10,101500.0,117450.0,103280.0,105530.0,111450.0,...,0.978881,0.051361,0.795917,0.656135,0.183007,0.078824,-0.000000,not,0.745592,6283
19607,YYS(ph)DS(ph)DDELTVEQR,1,BOD1L1,Q8NFC6,255.16,310360.0,366690.0,326720.0,360350.0,394830.0,...,1.003857,0.043479,2.968655,0.105661,0.976087,0.108746,-0.000000,not,0.680164,8770


Now Peptide IDs

We don't want to generate additional peptides for n terminal acetylation. So we replace N-Terminal ACs by Ms.

In [38]:
input_csv_df['Modified sequence'] = input_csv_df['Modified sequence'].apply(lambda seq: 'M' + seq[4:] if seq[0:4] == '(ac)' else seq)

In [39]:
input_csv_df['Sequence'] = input_csv_df['Modified sequence'].apply(
    lambda modseq: re.sub('\\s*\\(.*?\\)\\s*', '', modseq))

Create a temporary table to obtain the peptide ids for each of these sequences. Also retrieve the starting positions already, since we'll need those later.

In [40]:
seq_temp_table_name = f"temp_seq_{int(datetime.datetime.now().timestamp())}"
seq_temp_table_name

'temp_seq_1763734453'

In [41]:
with get_db_connection() as conn:
    input_csv_df[['PROTEIN_ID','Sequence']].drop_duplicates().to_sql(
        seq_temp_table_name,
        conn,
        if_exists='replace')
    #Create an index - speeds it up by 5-8 times
    conn.execute(f"CREATE INDEX idx_temp_seq ON {seq_temp_table_name}(Sequence)")

In [42]:
with get_db_connection() as conn:
    seq_to_peptideid_df = pd.read_sql(
       f"""
        SELECT P.*, PTP.START_POSITION
        FROM {seq_temp_table_name} T
        JOIN PEPTIDE P ON P.SEQUENCE = T.Sequence
        JOIN PROTEIN_TO_PEPTIDE PTP ON P.PEPTIDE_ID = PTP.PEPTIDE_ID AND T.PROTEIN_ID = PTP.PROTEIN_ID
        ;
        """,
        conn)
seq_to_peptideid_df.drop_duplicates()

,PEPTIDE_ID,SEQUENCE,START_POSITION
0,3068,TDEADAEERGPEENYSRPEAPNEFYDGDHDNDKESDVEI,390
1,3070,GPEENYSRPEAPNEFYDGDHDNDKESDVEI,399
2,3481,HTVLYISPPPEDLLDNSR,139
3,3714,EISSPARPCSFEEAMK,558
4,3960,KFELLPTPPLSPSR,52
...,...,...,...
16156,6171028,SALSSSLRDLSEAGVHH,798
16157,6198396,HSPSGMFDYDFEIDLK,177
16158,6203985,KQPPKEPSEVPTPK,31
16159,6203988,QPPKEPSEVPTPK,32


In [43]:
input_csv_df = input_csv_df.merge(
    seq_to_peptideid_df,
    left_on = 'Sequence',
    right_on = 'SEQUENCE',
    how='left').drop(['Sequence', 'SEQUENCE'], axis=1)

In [44]:
input_csv_df['PEPTIDE_ID'] = input_csv_df['PEPTIDE_ID'].astype('Int64')
input_csv_df['START_POSITION'] = input_csv_df['START_POSITION'].astype('Int64')
input_csv_df

,Modified sequence,N duplicates,Gene Names,Uniprot,Score,Raw 1,Raw 2,Raw 3,Raw 4,Raw 5,...,Curve F_Value,Curve P_Value,Curve Log P_Value,Curve F_Value SAM Corrected,Relevance Score,Regulation,Curve q-Value,PROTEIN_ID,PEPTIDE_ID,START_POSITION
0,MAAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,182.81,104000.0,108540.0,108270.0,129490.0,137980.0,...,1.474121,0.347278,0.459323,0.145907,0.000206,not,0.629313,12011,2984183,1
1,MAAAAAAAGDS(ph)DSWDADAFSVEDPVRK,6,EIF3J,O75822;O75822-2;O75822-3,233.50,325843.5,355756.6,337716.3,372103.7,376749.5,...,6.393457,0.017295,1.762080,0.088781,-0.000000,not,0.719848,12011,2984183,1
2,MAAAAAAAGDSDS(ph)WDADAFSVEDPVRK,3,EIF3J,O75822;O75822-2;O75822-3,170.56,546448.0,621476.0,573421.0,646003.0,671111.0,...,0.645374,0.751752,0.123925,0.015688,-0.000000,not,0.940467,12011,2984183,1
3,MAAAAAAAGDSDSWDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,116.35,42125.0,44422.0,41461.0,48380.0,47313.0,...,1.564582,0.320002,0.494848,0.082689,-0.000000,not,0.736682,12011,2984183,1
4,MAAAAPDSRVS(ph)EEENLKK,1,RRP15,Q9Y3B9,168.05,803650.0,889630.0,877020.0,925750.0,993160.0,...,1.329570,0.396697,0.401541,0.089889,-0.000000,not,0.717919,18595,4588708,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19774,YYGGGSEGGR,1,HNRNPL,P14866,177.78,108820.0,109800.0,104720.0,122130.0,121740.0,...,2.212674,0.184866,0.733144,0.299523,0.017102,not,0.500095,11525,2861969,47
19775,YYLHDDR,2,THRAP3,Q9Y2W1;A0A3B3ITZ9,158.36,452249.0,457444.0,425540.0,457611.0,497938.0,...,3.940836,0.057164,1.242875,0.034530,-0.000000,not,0.873684,13499,3349673,880
19776,YYRSPS(ph)R,1,RSRP1,H3BMX3;H3BTJ0;Q9BUV0-3;Q9BUV0-2;Q9BUV0,76.10,101500.0,117450.0,103280.0,105530.0,111450.0,...,0.795917,0.656135,0.183007,0.078824,-0.000000,not,0.745592,6283,1577496,104
19777,YYS(ph)DS(ph)DDELTVEQR,1,BOD1L1,Q8NFC6,255.16,310360.0,366690.0,326720.0,360350.0,394830.0,...,2.968655,0.105661,0.976087,0.108746,-0.000000,not,0.680164,8770,2192999,480


We do not import rows without a peptide id

In [45]:
input_csv_df = input_csv_df[pd.notna(input_csv_df['PEPTIDE_ID'])]
input_csv_df.reset_index(drop=True, inplace=True)
input_csv_df

,Modified sequence,N duplicates,Gene Names,Uniprot,Score,Raw 1,Raw 2,Raw 3,Raw 4,Raw 5,...,Curve F_Value,Curve P_Value,Curve Log P_Value,Curve F_Value SAM Corrected,Relevance Score,Regulation,Curve q-Value,PROTEIN_ID,PEPTIDE_ID,START_POSITION
0,MAAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,182.81,104000.0,108540.0,108270.0,129490.0,137980.0,...,1.474121,0.347278,0.459323,0.145907,0.000206,not,0.629313,12011,2984183,1
1,MAAAAAAAGDS(ph)DSWDADAFSVEDPVRK,6,EIF3J,O75822;O75822-2;O75822-3,233.50,325843.5,355756.6,337716.3,372103.7,376749.5,...,6.393457,0.017295,1.762080,0.088781,-0.000000,not,0.719848,12011,2984183,1
2,MAAAAAAAGDSDS(ph)WDADAFSVEDPVRK,3,EIF3J,O75822;O75822-2;O75822-3,170.56,546448.0,621476.0,573421.0,646003.0,671111.0,...,0.645374,0.751752,0.123925,0.015688,-0.000000,not,0.940467,12011,2984183,1
3,MAAAAAAAGDSDSWDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,116.35,42125.0,44422.0,41461.0,48380.0,47313.0,...,1.564582,0.320002,0.494848,0.082689,-0.000000,not,0.736682,12011,2984183,1
4,MAAAAPDSRVS(ph)EEENLKK,1,RRP15,Q9Y3B9,168.05,803650.0,889630.0,877020.0,925750.0,993160.0,...,1.329570,0.396697,0.401541,0.089889,-0.000000,not,0.717919,18595,4588708,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19431,YYGGGSEGGR,1,HNRNPL,P14866,177.78,108820.0,109800.0,104720.0,122130.0,121740.0,...,2.212674,0.184866,0.733144,0.299523,0.017102,not,0.500095,11525,2861969,47
19432,YYLHDDR,2,THRAP3,Q9Y2W1;A0A3B3ITZ9,158.36,452249.0,457444.0,425540.0,457611.0,497938.0,...,3.940836,0.057164,1.242875,0.034530,-0.000000,not,0.873684,13499,3349673,880
19433,YYRSPS(ph)R,1,RSRP1,H3BMX3;H3BTJ0;Q9BUV0-3;Q9BUV0-2;Q9BUV0,76.10,101500.0,117450.0,103280.0,105530.0,111450.0,...,0.795917,0.656135,0.183007,0.078824,-0.000000,not,0.745592,6283,1577496,104
19434,YYS(ph)DS(ph)DDELTVEQR,1,BOD1L1,Q8NFC6,255.16,310360.0,366690.0,326720.0,360350.0,394830.0,...,2.968655,0.105661,0.976087,0.108746,-0.000000,not,0.680164,8770,2192999,480


In [46]:
with get_db_connection() as conn:
    conn.execute(f"DROP TABLE {seq_temp_table_name}")

Create separate data frame for USER_DATUM_DETAIL

Log-transform fold changes if requested

In [47]:
#Find Fold Change column:
if request['parameters']['foldChangeDataFoldChangeScale'] == 'raw':
    for col in input_csv_df.columns:
        if col.lower().replace(' ', '') in ['foldchange', 'fc']:
            input_csv_df['Log Fold Change'] = np.log2(input_csv_df[col])
            break
    else:
        print('Trying to log-transform fold change column but could not find it!')

C:\Users\jmueller\AppData\Local\pypoetry\Cache\virtualenvs\standalone-backend-DmlZfqmW-py3.11\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\jmueller\AppData\Local\Temp\ipykernel_30380\2005389581.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  input_csv_df['Log Fold Change'] = np.log2(input_csv_df[col])


In [48]:
input_csv_df_detail = input_csv_df[detail_columns].stack().reset_index()
input_csv_df_detail.set_index('level_0', inplace=True)
input_csv_df_detail.columns = ['KEY', 'VALUE']
input_csv_df_detail

,KEY,VALUE
level_0,,
0,Curve R2,0.348977
1,Curve R2,0.699238
2,Curve R2,0.190074
3,Curve R2,0.362627
4,Curve R2,0.325909
...,...,...
19431,Curve R2,0.445863
19432,Curve R2,0.588990
19433,Curve R2,0.224460


We want to also store the modified sites as details. For this, we need to extract the site(s) from each peptide and then determine the position inside the protein

TODO: Determine the type of modification. Then either join on both identifier and modif type, or add the modif type to the identifier (e.g. as _ph suffix). But then you'd have to go back and change it in the modified_site table too.

In [49]:
def extract_modified_sites(row):
    res = []
    position_wo_mods = 0
    i = 0
    while i < len(row['Modified sequence']):
        if row['Modified sequence'][i] == '(':
            if i > 0:
                res.append(f"{row['UNIPROT_ACC']}_{row['Modified sequence'][i-1]}{position_wo_mods-1+row['START_POSITION']}")
                #Skip everything until closing parenthesis
                while i < len(row['Modified sequence']) and row['Modified sequence'][i] != ')':
                    i+=1
        else:
            position_wo_mods += 1
        i+=1
    return res

In [50]:
modsite_df = input_csv_df[['PROTEIN_ID','PEPTIDE_ID', 'START_POSITION', 'Modified sequence']].drop_duplicates().copy()
modsite_df = modsite_df.merge(protein_df[['PROTEIN_ID', 'UNIPROT_ACC']],
                 on='PROTEIN_ID',
                 how='left')

In [51]:
modsite_df['SITE_IDENTIFIER'] = modsite_df.apply(extract_modified_sites, axis=1)
modsite_df = modsite_df.explode('SITE_IDENTIFIER')
modsite_df

,PROTEIN_ID,PEPTIDE_ID,START_POSITION,Modified sequence,UNIPROT_ACC,SITE_IDENTIFIER
0,12011,2984183,1,MAAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,O75822,O75822_S11
0,12011,2984183,1,MAAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,O75822,O75822_S13
1,12011,2984183,1,MAAAAAAAGDS(ph)DSWDADAFSVEDPVRK,O75822,O75822_S11
2,12011,2984183,1,MAAAAAAAGDSDS(ph)WDADAFSVEDPVRK,O75822,O75822_S13
3,12011,2984183,1,MAAAAAAAGDSDSWDADAFSVEDPVRK,O75822,NaN
...,...,...,...,...,...,...
19358,13499,3349673,880,YYLHDDR,Q9Y2W1,NaN
19359,6283,1577496,104,YYRSPS(ph)R,Q9BUV0,Q9BUV0_S109
19360,8770,2192999,480,YYS(ph)DS(ph)DDELTVEQR,Q8NFC6,Q8NFC6_S482
19360,8770,2192999,480,YYS(ph)DS(ph)DDELTVEQR,Q8NFC6,Q8NFC6_S484


Now we again create a temporary table to get the MODIFIED_SITE_IDs for these SITE_IDENTIFIERs

In [52]:
modsite_temp_table_name = f"temp_modsite_{int(datetime.datetime.now().timestamp())}"
modsite_temp_table_name

'temp_modsite_1763734933'

In [53]:
with get_db_connection() as conn:
    modsite_df[['SITE_IDENTIFIER']].dropna().drop_duplicates().to_sql(
        modsite_temp_table_name,
        conn,
        if_exists='replace')
    conn.execute(f"CREATE INDEX idx_temp_modsite ON {modsite_temp_table_name}(SITE_IDENTIFIER)")

In [54]:
with get_db_connection() as conn:
    modsite_id_df = pd.read_sql(
       f"""
        SELECT M.MODIFIED_SITE_ID, M.SITE_IDENTIFIER
        FROM {modsite_temp_table_name} T
        JOIN MODIFIED_SITE M ON M.SITE_IDENTIFIER = T.SITE_IDENTIFIER
        ;
        """,
        conn)
modsite_id_df

,MODIFIED_SITE_ID,SITE_IDENTIFIER
0,5418593,A0A024R0Y4_S6
1,7297318,A0A087WT04_S127
2,6696088,A0A087WTW0_T38
3,235701,A0A087WV53_S38
4,235703,A0A087WV53_S44
...,...,...
14290,1761593,Q9Y6Y0_S326
14291,1761599,Q9Y6Y0_S338
14292,3865611,S5FZ81_S376
14293,4565218,V9GY78_S120


In [55]:
with get_db_connection() as conn:
    conn.execute(f"DROP TABLE {modsite_temp_table_name}")

In [56]:
modsite_df = modsite_df.reset_index().merge(modsite_id_df,how='left')
modsite_df

,index,PROTEIN_ID,PEPTIDE_ID,START_POSITION,Modified sequence,UNIPROT_ACC,SITE_IDENTIFIER,MODIFIED_SITE_ID
0,0,12011,2984183,1,MAAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,O75822,O75822_S11,1089281.0
1,0,12011,2984183,1,MAAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,O75822,O75822_S13,1089282.0
2,1,12011,2984183,1,MAAAAAAAGDS(ph)DSWDADAFSVEDPVRK,O75822,O75822_S11,1089281.0
3,2,12011,2984183,1,MAAAAAAAGDSDS(ph)WDADAFSVEDPVRK,O75822,O75822_S13,1089282.0
4,3,12011,2984183,1,MAAAAAAAGDSDSWDADAFSVEDPVRK,O75822,NaN,NaN
...,...,...,...,...,...,...,...,...
24217,19358,13499,3349673,880,YYLHDDR,Q9Y2W1,NaN,NaN
24218,19359,6283,1577496,104,YYRSPS(ph)R,Q9BUV0,Q9BUV0_S109,563379.0
24219,19360,8770,2192999,480,YYS(ph)DS(ph)DDELTVEQR,Q8NFC6,Q8NFC6_S482,789901.0
24220,19360,8770,2192999,480,YYS(ph)DS(ph)DDELTVEQR,Q8NFC6,Q8NFC6_S484,789902.0


In [57]:
modsite_df.set_index('index', inplace=True)
modsite_df['KEY'] = 'MODIFIED_SITE_ID'
modsite_df.rename({'MODIFIED_SITE_ID' : 'VALUE'}, axis=1, inplace=True,)
modsite_df[['KEY', 'VALUE']]

,KEY,VALUE
index,,
0,MODIFIED_SITE_ID,1089281.0
0,MODIFIED_SITE_ID,1089282.0
1,MODIFIED_SITE_ID,1089281.0
2,MODIFIED_SITE_ID,1089282.0
3,MODIFIED_SITE_ID,NaN
...,...,...
19358,MODIFIED_SITE_ID,NaN
19359,MODIFIED_SITE_ID,563379.0
19360,MODIFIED_SITE_ID,789901.0


In [58]:
input_csv_df['DATASET_ID'] = dataset_id

C:\Users\jmueller\AppData\Local\Temp\ipykernel_30380\2350025742.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  input_csv_df['DATASET_ID'] = dataset_id


In [59]:
if 'Experiment' not in input_csv_df:
    input_csv_df['Experiment'] = request['parameters']['datasetName']

C:\Users\jmueller\AppData\Local\Temp\ipykernel_30380\4227356843.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  input_csv_df['Experiment'] = request['parameters']['datasetName']


In [60]:
input_csv_df

,Modified sequence,N duplicates,Gene Names,Uniprot,Score,Raw 1,Raw 2,Raw 3,Raw 4,Raw 5,...,Curve F_Value SAM Corrected,Relevance Score,Regulation,Curve q-Value,PROTEIN_ID,PEPTIDE_ID,START_POSITION,Log Fold Change,DATASET_ID,Experiment
0,MAAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,182.81,104000.0,108540.0,108270.0,129490.0,137980.0,...,0.145907,0.000206,not,0.629313,12011,2984183,1,NaN,10,MyTestDecryptMDataset
1,MAAAAAAAGDS(ph)DSWDADAFSVEDPVRK,6,EIF3J,O75822;O75822-2;O75822-3,233.50,325843.5,355756.6,337716.3,372103.7,376749.5,...,0.088781,-0.000000,not,0.719848,12011,2984183,1,NaN,10,MyTestDecryptMDataset
2,MAAAAAAAGDSDS(ph)WDADAFSVEDPVRK,3,EIF3J,O75822;O75822-2;O75822-3,170.56,546448.0,621476.0,573421.0,646003.0,671111.0,...,0.015688,-0.000000,not,0.940467,12011,2984183,1,NaN,10,MyTestDecryptMDataset
3,MAAAAAAAGDSDSWDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,116.35,42125.0,44422.0,41461.0,48380.0,47313.0,...,0.082689,-0.000000,not,0.736682,12011,2984183,1,-3.583203,10,MyTestDecryptMDataset
4,MAAAAPDSRVS(ph)EEENLKK,1,RRP15,Q9Y3B9,168.05,803650.0,889630.0,877020.0,925750.0,993160.0,...,0.089889,-0.000000,not,0.717919,18595,4588708,1,-3.465410,10,MyTestDecryptMDataset
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19431,YYGGGSEGGR,1,HNRNPL,P14866,177.78,108820.0,109800.0,104720.0,122130.0,121740.0,...,0.299523,0.017102,not,0.500095,11525,2861969,47,NaN,10,MyTestDecryptMDataset
19432,YYLHDDR,2,THRAP3,Q9Y2W1;A0A3B3ITZ9,158.36,452249.0,457444.0,425540.0,457611.0,497938.0,...,0.034530,-0.000000,not,0.873684,13499,3349673,880,NaN,10,MyTestDecryptMDataset
19433,YYRSPS(ph)R,1,RSRP1,H3BMX3;H3BTJ0;Q9BUV0-3;Q9BUV0-2;Q9BUV0,76.10,101500.0,117450.0,103280.0,105530.0,111450.0,...,0.078824,-0.000000,not,0.745592,6283,1577496,104,NaN,10,MyTestDecryptMDataset
19434,YYS(ph)DS(ph)DDELTVEQR,1,BOD1L1,Q8NFC6,255.16,310360.0,366690.0,326720.0,360350.0,394830.0,...,0.108746,-0.000000,not,0.680164,8770,2192999,480,NaN,10,MyTestDecryptMDataset


Generate additional detail df with Curve Parameters

In [61]:
existing_additional_decryptM_columns = list(set(additional_expected_colnames_decryptM_map.keys()).intersection(input_csv_df.columns))
existing_additional_decryptM_columns

['Back',
 'Relevance Score',
 'Curve q-Value',
 'Regulation',
 'pEC50_Error',
 'pEC50',
 'Fold Change']

In [62]:
decryptM_details_df = input_csv_df[existing_additional_decryptM_columns].stack().reset_index()
decryptM_details_df.set_index('level_0', inplace=True)
decryptM_details_df.columns = ['KEY', 'VALUE']
decryptM_details_df

,KEY,VALUE
level_0,,
0,Back,2.170134
0,Relevance Score,0.000206
0,Curve q-Value,0.629313
0,Regulation,not
0,pEC50_Error,0.512414
...,...,...
19435,Curve q-Value,0.666263
19435,Regulation,not
19435,pEC50_Error,90.491936


Generate df for curve data  

In [63]:
toml_data = tomllib.load(open(request['file']['tomlFile'], 'rb'))

In [64]:
toml_data

{'experiments': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
 'doses': [0.0,
  0.03,
  0.3,
  1.0,
  3.0,
  10.0,
  30.0,
  100.0,
  300.0,
  1000.0,
  10000.0],
 'dose_scale': '1e-9',
 'dose_unit': 'M'}

In [65]:
for col in input_csv_df.columns:
        if any(f"{col.lower().replace(' ', '')}".startswith(expected) for expected in ['ratio','tmtratio','tmtchannelratio','lfqratio']):
            print(f'Found it: {col}')
            factor_col_prefix = " ".join(col.split()[:-1])
            break

Found it: Ratio 1


In [66]:
column_to_factor = {f"{factor_col_prefix} {exp}": "%.3g" % (fact*float(toml_data['dose_scale'])) 
                    for exp, fact in zip(toml_data['experiments'], toml_data['doses'])}
column_to_factor

{'Ratio 1': '0',
 'Ratio 2': '3e-11',
 'Ratio 3': '3e-10',
 'Ratio 4': '1e-09',
 'Ratio 5': '3e-09',
 'Ratio 6': '1e-08',
 'Ratio 7': '3e-08',
 'Ratio 8': '1e-07',
 'Ratio 9': '3e-07',
 'Ratio 10': '1e-06',
 'Ratio 11': '1e-05'}

In [67]:
ucd_df = input_csv_df[column_to_factor.keys()
    ].rename(column_to_factor, axis=1
            ).reset_index(
            ).melt(id_vars='index', var_name= 'FACTOR_VALUE', value_name='RESPONSE_VALUE'
                  ).set_index('index')
ucd_df

,FACTOR_VALUE,RESPONSE_VALUE
index,,
0,0,1.000000
1,0,1.000000
2,0,1.000000
3,0,1.000000
4,0,1.000000
...,...,...
19431,1e-05,0.840847
19432,1e-05,0.947980
19433,1e-05,0.963455


In [68]:
ucd_df['FACTOR_NAME'] = "Dose" #Hard coded for now, change once the toml file allows something else
ucd_df['FACTOR_UNIT'] = toml_data['dose_unit']
ucd_df

,FACTOR_VALUE,RESPONSE_VALUE,FACTOR_NAME,FACTOR_UNIT
index,,,,
0,0,1.000000,Dose,M
1,0,1.000000,Dose,M
2,0,1.000000,Dose,M
3,0,1.000000,Dose,M
4,0,1.000000,Dose,M
...,...,...,...,...
19431,1e-05,0.840847,Dose,M
19432,1e-05,0.947980,Dose,M
19433,1e-05,0.963455,Dose,M


In [69]:
with get_db_connection() as conn:
    next_user_datum_id = conn.execute('SELECT MAX(USER_DATUM_ID)+1 FROM USER_QUANTIFICATION_DATA').fetchall()[0][0]
    if not next_user_datum_id:
        next_user_datum_id = 1
    if request['parameters']['datasetType'] == 'Curve':
        next_user_curve_id = conn.execute('SELECT MAX(USER_CURVE_ID)+1 FROM USER_CURVE_DATA').fetchall()[0][0]
        if not next_user_curve_id:
           next_user_curve_id = 1 

In [70]:
input_csv_df.index+=next_user_datum_id
input_csv_df_detail.index+=next_user_datum_id
modsite_df.index+=next_user_datum_id

In [71]:
input_csv_df

,Modified sequence,N duplicates,Gene Names,Uniprot,Score,Raw 1,Raw 2,Raw 3,Raw 4,Raw 5,...,Curve F_Value SAM Corrected,Relevance Score,Regulation,Curve q-Value,PROTEIN_ID,PEPTIDE_ID,START_POSITION,Log Fold Change,DATASET_ID,Experiment
49742,MAAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,182.81,104000.0,108540.0,108270.0,129490.0,137980.0,...,0.145907,0.000206,not,0.629313,12011,2984183,1,NaN,10,MyTestDecryptMDataset
49743,MAAAAAAAGDS(ph)DSWDADAFSVEDPVRK,6,EIF3J,O75822;O75822-2;O75822-3,233.50,325843.5,355756.6,337716.3,372103.7,376749.5,...,0.088781,-0.000000,not,0.719848,12011,2984183,1,NaN,10,MyTestDecryptMDataset
49744,MAAAAAAAGDSDS(ph)WDADAFSVEDPVRK,3,EIF3J,O75822;O75822-2;O75822-3,170.56,546448.0,621476.0,573421.0,646003.0,671111.0,...,0.015688,-0.000000,not,0.940467,12011,2984183,1,NaN,10,MyTestDecryptMDataset
49745,MAAAAAAAGDSDSWDADAFSVEDPVRK,1,EIF3J,O75822;O75822-2;O75822-3,116.35,42125.0,44422.0,41461.0,48380.0,47313.0,...,0.082689,-0.000000,not,0.736682,12011,2984183,1,-3.583203,10,MyTestDecryptMDataset
49746,MAAAAPDSRVS(ph)EEENLKK,1,RRP15,Q9Y3B9,168.05,803650.0,889630.0,877020.0,925750.0,993160.0,...,0.089889,-0.000000,not,0.717919,18595,4588708,1,-3.465410,10,MyTestDecryptMDataset
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69173,YYGGGSEGGR,1,HNRNPL,P14866,177.78,108820.0,109800.0,104720.0,122130.0,121740.0,...,0.299523,0.017102,not,0.500095,11525,2861969,47,NaN,10,MyTestDecryptMDataset
69174,YYLHDDR,2,THRAP3,Q9Y2W1;A0A3B3ITZ9,158.36,452249.0,457444.0,425540.0,457611.0,497938.0,...,0.034530,-0.000000,not,0.873684,13499,3349673,880,NaN,10,MyTestDecryptMDataset
69175,YYRSPS(ph)R,1,RSRP1,H3BMX3;H3BTJ0;Q9BUV0-3;Q9BUV0-2;Q9BUV0,76.10,101500.0,117450.0,103280.0,105530.0,111450.0,...,0.078824,-0.000000,not,0.745592,6283,1577496,104,NaN,10,MyTestDecryptMDataset
69176,YYS(ph)DS(ph)DDELTVEQR,1,BOD1L1,Q8NFC6,255.16,310360.0,366690.0,326720.0,360350.0,394830.0,...,0.108746,-0.000000,not,0.680164,8770,2192999,480,NaN,10,MyTestDecryptMDataset


In [72]:
ucd_df

,FACTOR_VALUE,RESPONSE_VALUE,FACTOR_NAME,FACTOR_UNIT
index,,,,
0,0,1.000000,Dose,M
1,0,1.000000,Dose,M
2,0,1.000000,Dose,M
3,0,1.000000,Dose,M
4,0,1.000000,Dose,M
...,...,...,...,...
19431,1e-05,0.840847,Dose,M
19432,1e-05,0.947980,Dose,M
19433,1e-05,0.963455,Dose,M


In [73]:
next_user_curve_id

1

In [74]:
ucd_df

,FACTOR_VALUE,RESPONSE_VALUE,FACTOR_NAME,FACTOR_UNIT
index,,,,
0,0,1.000000,Dose,M
1,0,1.000000,Dose,M
2,0,1.000000,Dose,M
3,0,1.000000,Dose,M
4,0,1.000000,Dose,M
...,...,...,...,...
19431,1e-05,0.840847,Dose,M
19432,1e-05,0.947980,Dose,M
19433,1e-05,0.963455,Dose,M


In [75]:
if request['parameters']['datasetType'] == 'Curve':
    input_csv_df['USER_CURVE_ID'] = range(next_user_curve_id, next_user_curve_id + len(input_csv_df))
    decryptM_details_df.index += next_user_datum_id
    ucd_df.index += next_user_curve_id

C:\Users\jmueller\AppData\Local\Temp\ipykernel_30380\1851961017.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  input_csv_df['USER_CURVE_ID'] = range(next_user_curve_id, next_user_curve_id + len(input_csv_df))


In [76]:
uqd_df = input_csv_df.reset_index().rename(
{
    'index': 'USER_DATUM_ID',
    'Experiment': 'EXPERIMENT',
    'Regulation': 'REGULATION',
    'Modified sequence': 'MODIFIED_SEQUENCE'
}, axis=1)
uqd_df[['USER_DATUM_ID','DATASET_ID','REGULATION','PROTEIN_ID','PEPTIDE_ID', 'MODIFIED_SEQUENCE','EXPERIMENT']]

,USER_DATUM_ID,DATASET_ID,REGULATION,PROTEIN_ID,PEPTIDE_ID,MODIFIED_SEQUENCE,EXPERIMENT
0,49742,10,not,12011,2984183,MAAAAAAAGDS(ph)DS(ph)WDADAFSVEDPVRK,MyTestDecryptMDataset
1,49743,10,not,12011,2984183,MAAAAAAAGDS(ph)DSWDADAFSVEDPVRK,MyTestDecryptMDataset
2,49744,10,not,12011,2984183,MAAAAAAAGDSDS(ph)WDADAFSVEDPVRK,MyTestDecryptMDataset
3,49745,10,not,12011,2984183,MAAAAAAAGDSDSWDADAFSVEDPVRK,MyTestDecryptMDataset
4,49746,10,not,18595,4588708,MAAAAPDSRVS(ph)EEENLKK,MyTestDecryptMDataset
...,...,...,...,...,...,...,...
19431,69173,10,not,11525,2861969,YYGGGSEGGR,MyTestDecryptMDataset
19432,69174,10,not,13499,3349673,YYLHDDR,MyTestDecryptMDataset
19433,69175,10,not,6283,1577496,YYRSPS(ph)R,MyTestDecryptMDataset
19434,69176,10,not,8770,2192999,YYS(ph)DS(ph)DDELTVEQR,MyTestDecryptMDataset


In [77]:
with get_db_connection() as conn:
    if request['parameters']['datasetType'] == 'Curve':
            uqd_df[
                ['USER_DATUM_ID','DATASET_ID','REGULATION','USER_CURVE_ID','PROTEIN_ID','PEPTIDE_ID', 'MODIFIED_SEQUENCE','EXPERIMENT']
                ].to_sql(
                'USER_QUANTIFICATION_DATA',
                conn,
                if_exists='append',
                index=False)
            decryptM_details_df.reset_index(names='USER_DATUM_ID').to_sql(
                'USER_DATUM_DETAIL',
                conn,
                if_exists='append',
                index=False)
            ucd_df.reset_index(names='USER_CURVE_ID').to_sql(
                'USER_CURVE_DATA',
                conn,
                if_exists='append',
                index=False)
    else:
            uqd_df[
                ['USER_DATUM_ID','DATASET_ID','REGULATION','PROTEIN_ID','PEPTIDE_ID', 'MODIFIED_SEQUENCE','EXPERIMENT']
                ].to_sql(
                'USER_QUANTIFICATION_DATA',
                conn,
                if_exists='append',
                index=False)
        
    input_csv_df_detail.reset_index(names='USER_DATUM_ID').to_sql(
        'USER_DATUM_DETAIL',
        conn,
        if_exists='append',
        index=False)
    modsite_df.reset_index(names='USER_DATUM_ID')[
        ['USER_DATUM_ID', 'KEY', 'VALUE']
        ].to_sql(
            'USER_DATUM_DETAIL',
            conn,
            if_exists='append',
            index=False)

## B. Site

A lot of stuff will be duplicated

In [412]:
request = {
    'parameters': {
        'uuid': 'B3476C32AD2B4CCF8FB44BF9814C387F',
        'datasetName':'MyTestSiteDataset',
        'datasetType': 'FoldChange',#Curve/FoldChange
        'omics': 'Phosphorylation',#Phosphorylation/Other/Protein #TODO: Map to OMIC table instead of storing string value
        'foldChangeDataFoldChangeScale': 'raw', #raw/log/none
        'taxcode': 9606, #9606/10090
        
    },
    'file': {
        #TODO: See if this is possible or if you can only do a list
        'csvFile': '/home/jmueller/Downloads/examples/fold_change_data/psitelevel_data.csv',
    }
}
request


{'parameters': {'uuid': 'B3476C32AD2B4CCF8FB44BF9814C387F',
  'datasetName': 'MyTestSiteDataset',
  'datasetType': 'FoldChange',
  'omics': 'Phosphorylation',
  'foldChangeDataFoldChangeScale': 'raw',
  'taxcode': 9606},
 'file': {'csvFile': '/home/jmueller/Downloads/examples/fold_change_data/psitelevel_data.csv'}}

In [413]:
user_id = getOrCreateUserId(request['parameters']['uuid'])
user_id

5

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [414]:
dataset_id = insertToUserDataset(
    request['parameters']['datasetName'],
    user_id,
    request['parameters']['datasetType'],
    request['parameters']['omics'],
    request['parameters']['taxcode'],
)
dataset_id

6

In [415]:
csv_delimiter = get_delimiter(request['file']['csvFile'], 50000)

In [416]:
input_csv_df = pd.read_csv(request['file']['csvFile'], sep=csv_delimiter)
input_csv_df

,Protein IDs,p-site,Log Fold Change,Adjusted pvalue,Regulation
0,O75822,S11,-5.345297,0.458331,NaN
1,O75822,S13,-0.361372,0.967921,NaN
2,Q9UID3,S18,0.767573,0.906945,NaN
3,Q01167,S30,-0.768159,0.657085,NaN
4,Q01167,S9,-0.768159,0.657085,NaN
...,...,...,...,...,...
10272,Q15678,S593,0.105275,0.851965,NaN
10273,Q15678,S594,-0.338962,0.372046,NaN
10274,P51784,S648,-0.443683,0.591849,NaN
10275,Q9H6S3,S570,-2.232358,0.562901,NaN


In [423]:
detail_columns = []
actual_colnames_to_expected_colnames = {}
for col in input_csv_df.columns:
    expected_colname_for_actual_colname = colnames_variants_lookup.get(col.lower().replace(' ', ''))
    if expected_colname_for_actual_colname:
        actual_colnames_to_expected_colnames[col] = expected_colname_for_actual_colname
    else:
        detail_columns.append(col)

In [425]:
input_csv_df.rename(actual_colnames_to_expected_colnames, axis=1, inplace=True)
input_csv_df

,Uniprot,Psite,Log Fold Change,Adjusted pvalue,Regulation
0,O75822,S11,-5.345297,0.458331,NaN
1,O75822,S13,-0.361372,0.967921,NaN
2,Q9UID3,S18,0.767573,0.906945,NaN
3,Q01167,S30,-0.768159,0.657085,NaN
4,Q01167,S9,-0.768159,0.657085,NaN
...,...,...,...,...,...
10272,Q15678,S593,0.105275,0.851965,NaN
10273,Q15678,S594,-0.338962,0.372046,NaN
10274,P51784,S648,-0.443683,0.591849,NaN
10275,Q9H6S3,S570,-2.232358,0.562901,NaN


In [426]:
input_csv_df['Regulation'] = input_csv_df['Regulation'].apply(clean_regulation)
input_csv_df

,Uniprot,Psite,Log Fold Change,Adjusted pvalue,Regulation
0,O75822,S11,-5.345297,0.458331,None
1,O75822,S13,-0.361372,0.967921,None
2,Q9UID3,S18,0.767573,0.906945,None
3,Q01167,S30,-0.768159,0.657085,None
4,Q01167,S9,-0.768159,0.657085,None
...,...,...,...,...,...
10272,Q15678,S593,0.105275,0.851965,None
10273,Q15678,S594,-0.338962,0.372046,None
10274,P51784,S648,-0.443683,0.591849,None
10275,Q9H6S3,S570,-2.232358,0.562901,None


In [427]:
'Psite' in input_csv_df

True

So we skip clean sequence

TODO: What if there is no Uniprot column? Then skip directly to Gene Name

In [435]:
if 'Uniprot' in input_csv_df:
    print('yay')

yay


In [429]:
uniprot_splits = input_csv_df['Uniprot'].apply(lambda s: re.split(';|,', s)).explode()
uniprot_splits = pd.DataFrame(uniprot_splits)

In [ ]:
uniprot_splits = uniprot_splits.reset_index(
).merge(protein_df[['PROTEIN_ID', 'UNIPROT_ACC']],
                   left_on = 'Uniprot',
                   right_on = 'UNIPROT_ACC',
                   how='left'
                  ).dropna(
                  ).drop(['Uniprot', 'UNIPROT_ACC'], axis=1
                        ).rename({'PROTEIN_ID':'PROTEIN_ID_FROM_UNIPROT'},
                                axis=1)
uniprot_splits

In [431]:
unique_mapped_proteins_after_uniprot = uniprot_splits.groupby('index').agg('min')
unique_mapped_proteins_after_uniprot

,PROTEIN_ID_FROM_UNIPROT
index,
0,12011.0
1,12011.0
2,5097.0
3,10232.0
4,10232.0
...,...
10272,19845.0
10273,19845.0
10274,2123.0


In [432]:
input_csv_df = input_csv_df.merge(unique_mapped_proteins_after_uniprot,
                   left_index=True,
                   right_index=True,
                   how="left")
input_csv_df

,Uniprot,Psite,Log Fold Change,Adjusted pvalue,Regulation,PROTEIN_ID_FROM_UNIPROT
0,O75822,S11,-5.345297,0.458331,None,12011.0
1,O75822,S13,-0.361372,0.967921,None,12011.0
2,Q9UID3,S18,0.767573,0.906945,None,5097.0
3,Q01167,S30,-0.768159,0.657085,None,10232.0
4,Q01167,S9,-0.768159,0.657085,None,10232.0
...,...,...,...,...,...,...
10272,Q15678,S593,0.105275,0.851965,None,19845.0
10273,Q15678,S594,-0.338962,0.372046,None,19845.0
10274,P51784,S648,-0.443683,0.591849,None,2123.0
10275,Q9H6S3,S570,-2.232358,0.562901,None,14187.0


In [437]:
#TODO: Auch das geht nur wenn es UNiprot gab. Was wenn es das nicht gab?
#TODO: Ergo: Stelle sicher dass es mindestens eines davon gibt sonst kannst du keine PROTEIN IDs mappen
#Glaube das muss Gene Names sein. Hat whrshl nur funktioniert weil du es spaeter eingefuegt hast und es dann die spalte schon gab.
if 'Gene Name' in input_csv_df:
    unmapped_proteins_after_uniprot = input_csv_df[pd.isna(input_csv_df['PROTEIN_ID_FROM_UNIPROT'])][['Gene Names']].dropna().drop_duplicates().copy()
    unmapped_proteins_after_uniprot['Gene Name'] = unmapped_proteins_after_uniprot['Gene Names'].dropna().apply(lambda s: re.split(';|,', s))
    unmapped_proteins_after_uniprot = unmapped_proteins_after_uniprot.explode('Gene Name')
    mapped_proteins_after_genename = unmapped_proteins_after_uniprot.reset_index(
        ).merge(protein_df[['PROTEIN_ID', 'GENE_NAME']],
                           left_on = 'Gene Name',
                           right_on = 'GENE_NAME',
                           how='left'
        ).drop(['Gene Name', 'GENE_NAME', 'Gene Names'], axis=1
        ).rename({'PROTEIN_ID':'PROTEIN_ID_FROM_GENE_NAME'},
                                        axis=1).dropna()
    #To resolve ambiguities, group by row index and retain only the smallest protein id 
    #(since canonical proteins were inserted first, this should be the most canonical isoform possible)
    unique_mapped_proteins_after_genename = mapped_proteins_after_genename.groupby('index').agg('min')
    input_csv_df = input_csv_df.merge(
        unique_mapped_proteins_after_genename,
        left_index=True,
        right_index=True,
        how='left')

In [440]:
input_csv_df

,Uniprot,Psite,Log Fold Change,Adjusted pvalue,Regulation,PROTEIN_ID
0,O75822,S11,-5.345297,0.458331,None,12011.0
1,O75822,S13,-0.361372,0.967921,None,12011.0
2,Q9UID3,S18,0.767573,0.906945,None,5097.0
3,Q01167,S30,-0.768159,0.657085,None,10232.0
4,Q01167,S9,-0.768159,0.657085,None,10232.0
...,...,...,...,...,...,...
10272,Q15678,S593,0.105275,0.851965,None,19845.0
10273,Q15678,S594,-0.338962,0.372046,None,19845.0
10274,P51784,S648,-0.443683,0.591849,None,2123.0
10275,Q9H6S3,S570,-2.232358,0.562901,None,14187.0


In [ ]:
if 'PROTEIN_ID_FROM_UNIPROT' in input_csv_df and 'PROTEIN_ID_FROM_GENE_NAME' in input_csv_df:
    input_csv_df['PROTEIN_ID'] = input_csv_df.apply(clean_protein_id, axis=1).astype('Int64')
    input_csv_df.drop(['PROTEIN_ID_FROM_UNIPROT', 'PROTEIN_ID_FROM_GENE_NAME'], axis=1, inplace=True)
elif 'PROTEIN_ID_FROM_UNIPROT' in input_csv_df:
    input_csv_df.rename({'PROTEIN_ID_FROM_UNIPROT': 'PROTEIN_ID'}, axis=1, inplace=True)
elif 'PROTEIN_ID_FROM_GENE_NAME' in input_csv_df:
    input_csv_df.rename({'PROTEIN_ID_FROM_GENE_NAME': 'PROTEIN_ID'}, axis=1, inplace=True)

Creating proper site identifiers  
In order to be totally sure we have the right Uniprots, let's map to UNIPROT_ACC from the PROTEIN table

In [447]:
input_csv_df = input_csv_df.merge(
    protein_df[['PROTEIN_ID', 'UNIPROT_ACC']],
    left_on='PROTEIN_ID',
    right_on='PROTEIN_ID',
    how='left')

In [451]:
input_csv_df['SITE_IDENTIFIER'] = input_csv_df.apply(lambda row: f'{row['UNIPROT_ACC']}_{row['Psite']}', axis=1)
input_csv_df

,Uniprot,Psite,Log Fold Change,Adjusted pvalue,Regulation,PROTEIN_ID,UNIPROT_ACC,SITE_IDENTIFIER
0,O75822,S11,-5.345297,0.458331,None,12011.0,O75822,O75822_S11
1,O75822,S13,-0.361372,0.967921,None,12011.0,O75822,O75822_S13
2,Q9UID3,S18,0.767573,0.906945,None,5097.0,Q9UID3,Q9UID3_S18
3,Q01167,S30,-0.768159,0.657085,None,10232.0,Q01167,Q01167_S30
4,Q01167,S9,-0.768159,0.657085,None,10232.0,Q01167,Q01167_S9
...,...,...,...,...,...,...,...,...
10272,Q15678,S593,0.105275,0.851965,None,19845.0,Q15678,Q15678_S593
10273,Q15678,S594,-0.338962,0.372046,None,19845.0,Q15678,Q15678_S594
10274,P51784,S648,-0.443683,0.591849,None,2123.0,P51784,P51784_S648
10275,Q9H6S3,S570,-2.232358,0.562901,None,14187.0,Q9H6S3,Q9H6S3_S570


Do the temp table thing

In [460]:
modsite_temp_table_name = f"temp_modsite_{int(datetime.datetime.now().timestamp())}"
with get_db_connection() as conn:
    input_csv_df[['SITE_IDENTIFIER']].dropna().drop_duplicates().to_sql(
        modsite_temp_table_name,
        conn,
        if_exists='replace')
    conn.execute(f"CREATE INDEX idx_temp_modsite ON {modsite_temp_table_name}(SITE_IDENTIFIER)")

In [457]:
with get_db_connection() as conn:
    modsite_id_df = pd.read_sql(
       f"""
        SELECT M.MODIFIED_SITE_ID, M.SITE_IDENTIFIER
        FROM {modsite_temp_table_name} T
        JOIN MODIFIED_SITE M ON M.SITE_IDENTIFIER = T.SITE_IDENTIFIER
        ;
        """,
        conn)
modsite_id_df

,MODIFIED_SITE_ID,SITE_IDENTIFIER
0,116546,A0FGR8_S693
1,116547,A0FGR8_S699
2,116549,A0FGR8_S704
3,116554,A0FGR8_S738
4,116555,A0FGR8_S739
...,...,...
8372,463012,Q9Y6X9_S739
8373,463013,Q9Y6X9_S743
8374,462898,Q9Y6X9_T4
8375,462979,Q9Y6X9_T565


In [476]:
with get_db_connection() as conn:
    conn.execute(f"DROP TABLE {modsite_temp_table_name}")

In [467]:
input_csv_df = input_csv_df.reset_index().merge(modsite_id_df,how='left')
input_csv_df

,index,Uniprot,Psite,Log Fold Change,Adjusted pvalue,Regulation,PROTEIN_ID,UNIPROT_ACC,SITE_IDENTIFIER,MODIFIED_SITE_ID
0,0,O75822,S11,-5.345297,0.458331,None,12011.0,O75822,O75822_S11,1089281.0
1,1,O75822,S13,-0.361372,0.967921,None,12011.0,O75822,O75822_S13,1089282.0
2,2,Q9UID3,S18,0.767573,0.906945,None,5097.0,Q9UID3,Q9UID3_S18,454756.0
3,3,Q01167,S30,-0.768159,0.657085,None,10232.0,Q01167,Q01167_S30,922562.0
4,4,Q01167,S9,-0.768159,0.657085,None,10232.0,Q01167,Q01167_S9,922560.0
...,...,...,...,...,...,...,...,...,...,...
10272,10272,Q15678,S593,0.105275,0.851965,None,19845.0,Q15678,Q15678_S593,1806987.0
10273,10273,Q15678,S594,-0.338962,0.372046,None,19845.0,Q15678,Q15678_S594,1806988.0
10274,10274,P51784,S648,-0.443683,0.591849,None,2123.0,P51784,P51784_S648,193261.0
10275,10275,Q9H6S3,S570,-2.232358,0.562901,None,14187.0,Q9H6S3,Q9H6S3_S570,1287069.0


In [474]:
input_csv_df = input_csv_df[pd.notna(input_csv_df['MODIFIED_SITE_ID'])]
input_csv_df['MODIFIED_SITE_ID'] = input_csv_df['MODIFIED_SITE_ID'].astype('Int64')

/tmp/ipykernel_2800565/975500686.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  input_csv_df['MODIFIED_SITE_ID'] = input_csv_df['MODIFIED_SITE_ID'].astype('Int64')


In [475]:
input_csv_df

,index,Uniprot,Psite,Log Fold Change,Adjusted pvalue,Regulation,PROTEIN_ID,UNIPROT_ACC,SITE_IDENTIFIER,MODIFIED_SITE_ID
0,0,O75822,S11,-5.345297,0.458331,None,12011.0,O75822,O75822_S11,1089281
1,1,O75822,S13,-0.361372,0.967921,None,12011.0,O75822,O75822_S13,1089282
2,2,Q9UID3,S18,0.767573,0.906945,None,5097.0,Q9UID3,Q9UID3_S18,454756
3,3,Q01167,S30,-0.768159,0.657085,None,10232.0,Q01167,Q01167_S30,922562
4,4,Q01167,S9,-0.768159,0.657085,None,10232.0,Q01167,Q01167_S9,922560
...,...,...,...,...,...,...,...,...,...,...
10272,10272,Q15678,S593,0.105275,0.851965,None,19845.0,Q15678,Q15678_S593,1806987
10273,10273,Q15678,S594,-0.338962,0.372046,None,19845.0,Q15678,Q15678_S594,1806988
10274,10274,P51784,S648,-0.443683,0.591849,None,2123.0,P51784,P51784_S648,193261
10275,10275,Q9H6S3,S570,-2.232358,0.562901,None,14187.0,Q9H6S3,Q9H6S3_S570,1287069


In [477]:
input_csv_df_detail = input_csv_df[detail_columns].stack().reset_index()
input_csv_df_detail.set_index('level_0', inplace=True)
input_csv_df_detail.columns = ['KEY', 'VALUE']
input_csv_df_detail

,KEY,VALUE
level_0,,
0,Log Fold Change,-5.345297
0,Adjusted pvalue,0.458331
1,Log Fold Change,-0.361372
1,Adjusted pvalue,0.967921
2,Log Fold Change,0.767573
...,...,...
10274,Adjusted pvalue,0.591849
10275,Log Fold Change,-2.232358
10275,Adjusted pvalue,0.562901


In [478]:
input_csv_df['DATASET_ID'] = dataset_id
if 'Experiment' not in input_csv_df:
    input_csv_df['Experiment'] = request['parameters']['datasetName']
input_csv_df

/tmp/ipykernel_2800565/287951026.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  input_csv_df['DATASET_ID'] = dataset_id
/tmp/ipykernel_2800565/287951026.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  input_csv_df['Experiment'] = request['parameters']['datasetName']


,index,Uniprot,Psite,Log Fold Change,Adjusted pvalue,Regulation,PROTEIN_ID,UNIPROT_ACC,SITE_IDENTIFIER,MODIFIED_SITE_ID,DATASET_ID,Experiment
0,0,O75822,S11,-5.345297,0.458331,None,12011.0,O75822,O75822_S11,1089281,6,MyTestSiteDataset
1,1,O75822,S13,-0.361372,0.967921,None,12011.0,O75822,O75822_S13,1089282,6,MyTestSiteDataset
2,2,Q9UID3,S18,0.767573,0.906945,None,5097.0,Q9UID3,Q9UID3_S18,454756,6,MyTestSiteDataset
3,3,Q01167,S30,-0.768159,0.657085,None,10232.0,Q01167,Q01167_S30,922562,6,MyTestSiteDataset
4,4,Q01167,S9,-0.768159,0.657085,None,10232.0,Q01167,Q01167_S9,922560,6,MyTestSiteDataset
...,...,...,...,...,...,...,...,...,...,...,...,...
10272,10272,Q15678,S593,0.105275,0.851965,None,19845.0,Q15678,Q15678_S593,1806987,6,MyTestSiteDataset
10273,10273,Q15678,S594,-0.338962,0.372046,None,19845.0,Q15678,Q15678_S594,1806988,6,MyTestSiteDataset
10274,10274,P51784,S648,-0.443683,0.591849,None,2123.0,P51784,P51784_S648,193261,6,MyTestSiteDataset
10275,10275,Q9H6S3,S570,-2.232358,0.562901,None,14187.0,Q9H6S3,Q9H6S3_S570,1287069,6,MyTestSiteDataset


In [491]:
#Not sure if I have the index at this point but I think not
modsite_df = input_csv_df.reset_index()[['index', 'MODIFIED_SITE_ID']]
modsite_df['KEY'] = 'MODIFIED_SITE_ID'

In [501]:
modsite_df.rename({'MODIFIED_SITE_ID' : 'VALUE'}, axis=1, inplace=True)
modsite_df

,index,VALUE,KEY
0,0,1089281,MODIFIED_SITE_ID
1,1,1089282,MODIFIED_SITE_ID
2,2,454756,MODIFIED_SITE_ID
3,3,922562,MODIFIED_SITE_ID
4,4,922560,MODIFIED_SITE_ID
...,...,...,...
10146,10272,1806987,MODIFIED_SITE_ID
10147,10273,1806988,MODIFIED_SITE_ID
10148,10274,193261,MODIFIED_SITE_ID
10149,10275,1287069,MODIFIED_SITE_ID


In [479]:
with get_db_connection() as conn:
    next_user_datum_id = conn.execute('SELECT MAX(USER_DATUM_ID)+1 FROM USER_QUANTIFICATION_DATA').fetchall()[0][0]
    if not next_user_datum_id:
        next_user_datum_id = 1
    if request['parameters']['datasetType'] == 'Curve':
        next_user_curve_id = conn.execute('SELECT MAX(USER_CURVE_ID)+1 FROM USER_CURVE_DATA').fetchall()[0][0]
        if not next_user_curve_id:
           next_user_curve_id = 1 

In [504]:
input_csv_df.index+=next_user_datum_id
input_csv_df_detail.index+=next_user_datum_id
modsite_df.index+=next_user_datum_id

In [507]:
modsite_df

,index,VALUE,KEY
38372,0,1089281,MODIFIED_SITE_ID
38373,1,1089282,MODIFIED_SITE_ID
38374,2,454756,MODIFIED_SITE_ID
38375,3,922562,MODIFIED_SITE_ID
38376,4,922560,MODIFIED_SITE_ID
...,...,...,...
48518,10272,1806987,MODIFIED_SITE_ID
48519,10273,1806988,MODIFIED_SITE_ID
48520,10274,193261,MODIFIED_SITE_ID
48521,10275,1287069,MODIFIED_SITE_ID


In [508]:
uqd_df = input_csv_df.reset_index().rename(
{
    'index': 'USER_DATUM_ID',
    'Experiment': 'EXPERIMENT',
    'Regulation': 'REGULATION',
}, axis=1)
uqd_df
#uqd_df[['USER_DATUM_ID','DATASET_ID','REGULATION','PROTEIN_ID','PEPTIDE_ID', 'MODIFIED_SEQUENCE','EXPERIMENT']]

,USER_DATUM_ID,Uniprot,Psite,Log Fold Change,Adjusted pvalue,REGULATION,PROTEIN_ID,UNIPROT_ACC,SITE_IDENTIFIER,MODIFIED_SITE_ID,DATASET_ID,EXPERIMENT
0,38372,O75822,S11,-5.345297,0.458331,None,12011.0,O75822,O75822_S11,1089281,6,MyTestSiteDataset
1,38373,O75822,S13,-0.361372,0.967921,None,12011.0,O75822,O75822_S13,1089282,6,MyTestSiteDataset
2,38374,Q9UID3,S18,0.767573,0.906945,None,5097.0,Q9UID3,Q9UID3_S18,454756,6,MyTestSiteDataset
3,38375,Q01167,S30,-0.768159,0.657085,None,10232.0,Q01167,Q01167_S30,922562,6,MyTestSiteDataset
4,38376,Q01167,S9,-0.768159,0.657085,None,10232.0,Q01167,Q01167_S9,922560,6,MyTestSiteDataset
...,...,...,...,...,...,...,...,...,...,...,...,...
10146,48644,Q15678,S593,0.105275,0.851965,None,19845.0,Q15678,Q15678_S593,1806987,6,MyTestSiteDataset
10147,48645,Q15678,S594,-0.338962,0.372046,None,19845.0,Q15678,Q15678_S594,1806988,6,MyTestSiteDataset
10148,48646,P51784,S648,-0.443683,0.591849,None,2123.0,P51784,P51784_S648,193261,6,MyTestSiteDataset
10149,48647,Q9H6S3,S570,-2.232358,0.562901,None,14187.0,Q9H6S3,Q9H6S3_S570,1287069,6,MyTestSiteDataset


In [515]:
#TODO: Slightly different columns bc no peptide id and mod seq and 
with get_db_connection() as conn:
    if request['parameters']['datasetType'] == 'Curve':
            uqd_df[
                ['USER_DATUM_ID','DATASET_ID','REGULATION','USER_CURVE_ID','PROTEIN_ID','EXPERIMENT']
                ].to_sql(
                'USER_QUANTIFICATION_DATA',
                conn,
                if_exists='append',
                index=False)
            decryptM_details_df.reset_index(names='USER_DATUM_ID').to_sql(
                'USER_DATUM_DETAIL',
                conn,
                if_exists='append',
                index=False)
            ucd_df.reset_index(names='USER_CURVE_ID').to_sql(
                'USER_CURVE_DATA',
                conn,
                if_exists='append',
                index=False)
    else:
            uqd_df[
                ['USER_DATUM_ID','DATASET_ID','REGULATION','PROTEIN_ID','EXPERIMENT']
                ].to_sql(
                'USER_QUANTIFICATION_DATA',
                conn,
                if_exists='append',
                index=False)
        
    input_csv_df_detail.reset_index(names='USER_DATUM_ID').to_sql(
        'USER_DATUM_DETAIL',
        conn,
        if_exists='append',
        index=False)
    modsite_df.reset_index(names='USER_DATUM_ID')[
        ['USER_DATUM_ID', 'KEY', 'VALUE']
        ].to_sql(
            'USER_DATUM_DETAIL',
            conn,
            if_exists='append',
            index=False)

# II. Protein

Agaaaaain a lot of duplicication

In [3]:
request = {
    'parameters': {
        'uuid': 'B3476C32AD2B4CCF8FB44BF9814C387F',
        'datasetName':'MyTestProteinDataset',
        'datasetType': 'FoldChange',#Curve/FoldChange
        'omics': 'Protein',#Phosphorylation/Other/Protein #TODO: Map to OMIC table instead of storing string value
        'foldChangeDataFoldChangeScale': 'raw', #raw/log/none
        'taxcode': 9606, #9606/10090
        
    },
    'file': {
        #TODO: See if this is possible or if you can only do a list
        'csvFile': '/home/jmueller/Downloads/examples/fold_change_data/volcano_fp_data.csv',
    }
}
request


{'parameters': {'uuid': 'B3476C32AD2B4CCF8FB44BF9814C387F',
  'datasetName': 'MyTestProteinDataset',
  'datasetType': 'FoldChange',
  'omics': 'Protein',
  'foldChangeDataFoldChangeScale': 'raw',
  'taxcode': 9606},
 'file': {'csvFile': '/home/jmueller/Downloads/examples/fold_change_data/volcano_fp_data.csv'}}

In [8]:
user_id = getOrCreateUserId(request['parameters']['uuid'])
user_id

5

In [10]:
dataset_id = insertToUserDataset(
    request['parameters']['datasetName'],
    user_id,
    request['parameters']['datasetType'],
    request['parameters']['omics'],
    request['parameters']['taxcode'],
)
dataset_id

7

In [12]:
csv_delimiter = get_delimiter(request['file']['csvFile'], 50000)
input_csv_df = pd.read_csv(request['file']['csvFile'], sep=csv_delimiter)
input_csv_df

,Gene names,Protein IDs,Log Fold Change,Adjusted pvalue,Regulation
0,HNRNPL,P14866,-0.641359,0.646447,NaN
1,EPS8L2,Q9H6S3,-2.232358,0.562901,NaN
2,USP11,P51784,-0.443683,0.591849,NaN
3,PTPN14,Q15678,-0.338962,0.372046,NaN
4,ATRX,P46100,-0.898034,0.722569,NaN
...,...,...,...,...,...
2489,LRCH4,O75427,-1.837674,0.665034,NaN
2490,TOMM22,Q9NS69,-0.769022,0.523625,NaN
2491,TRAF2,Q12933,2.494855,0.206656,NaN
2492,ZNF579,Q8NAF0,-1.132761,0.629816,NaN


In [15]:
detail_columns = []
actual_colnames_to_expected_colnames = {}
for col in input_csv_df.columns:
    expected_colname_for_actual_colname = colnames_variants_lookup.get(col.lower().replace(' ', ''))
    if expected_colname_for_actual_colname:
        actual_colnames_to_expected_colnames[col] = expected_colname_for_actual_colname
    else:
        detail_columns.append(col)

In [16]:
input_csv_df.rename(actual_colnames_to_expected_colnames, axis=1, inplace=True)
input_csv_df

,Gene Names,Uniprot,Log Fold Change,Adjusted pvalue,Regulation
0,HNRNPL,P14866,-0.641359,0.646447,NaN
1,EPS8L2,Q9H6S3,-2.232358,0.562901,NaN
2,USP11,P51784,-0.443683,0.591849,NaN
3,PTPN14,Q15678,-0.338962,0.372046,NaN
4,ATRX,P46100,-0.898034,0.722569,NaN
...,...,...,...,...,...
2489,LRCH4,O75427,-1.837674,0.665034,NaN
2490,TOMM22,Q9NS69,-0.769022,0.523625,NaN
2491,TRAF2,Q12933,2.494855,0.206656,NaN
2492,ZNF579,Q8NAF0,-1.132761,0.629816,NaN


In [19]:
input_csv_df['Regulation'] = input_csv_df['Regulation'].apply(clean_regulation)
input_csv_df

,Gene Names,Uniprot,Log Fold Change,Adjusted pvalue,Regulation
0,HNRNPL,P14866,-0.641359,0.646447,None
1,EPS8L2,Q9H6S3,-2.232358,0.562901,None
2,USP11,P51784,-0.443683,0.591849,None
3,PTPN14,Q15678,-0.338962,0.372046,None
4,ATRX,P46100,-0.898034,0.722569,None
...,...,...,...,...,...
2489,LRCH4,O75427,-1.837674,0.665034,None
2490,TOMM22,Q9NS69,-0.769022,0.523625,None
2491,TRAF2,Q12933,2.494855,0.206656,None
2492,ZNF579,Q8NAF0,-1.132761,0.629816,None


In [20]:
with get_db_connection() as conn:
    protein_df = pd.read_sql('SELECT PROTEIN_ID, GENE_NAME, UNIPROT_ACC FROM PROTEIN WHERE TAXCODE = ?',
                             conn,
                             params=[request['parameters']['taxcode']]) 
protein_df

,PROTEIN_ID,GENE_NAME,UNIPROT_ACC
0,1,TRBV18,A0A087X0M5
1,2,TMEM247,A6NEH6
2,3,UNC119B,A6NIH7
3,4,None,A6NJR5
4,5,TMEM278,A6NKF7
...,...,...,...
105714,105715,None,A0A0D9SG52
105715,105716,None,A0A1W2PRQ8
105716,105717,None,C9J4A7
105717,105718,None,G3V3Y1


In [23]:
uniprot_splits = input_csv_df['Uniprot'].apply(lambda s: re.split(';|,', s)).explode()
uniprot_splits = pd.DataFrame(uniprot_splits)
uniprot_splits = uniprot_splits.reset_index(
).merge(protein_df[['PROTEIN_ID', 'UNIPROT_ACC']],
                   left_on = 'Uniprot',
                   right_on = 'UNIPROT_ACC',
                   how='left'
                  ).dropna(
                  ).drop(['Uniprot', 'UNIPROT_ACC'], axis=1
                        ).rename({'PROTEIN_ID':'PROTEIN_ID_FROM_UNIPROT'},
                                axis=1)
unique_mapped_proteins_after_uniprot = uniprot_splits.groupby('index').agg('min')
unique_mapped_proteins_after_uniprot
input_csv_df = input_csv_df.merge(unique_mapped_proteins_after_uniprot,
                   left_index=True,
                   right_index=True,
                   how="left")

In [25]:
#TODO: Auch das geht nur wenn es UNiprot gab. Was wenn es das nicht gab?
#TODO: Ergo: Stelle sicher dass es mindestens eines davon gibt sonst kannst du keine PROTEIN IDs mappen
if 'Gene Name' in input_csv_df:
    unmapped_proteins_after_uniprot = input_csv_df[pd.isna(input_csv_df['PROTEIN_ID_FROM_UNIPROT'])][['Gene Names']].dropna().drop_duplicates().copy()
    unmapped_proteins_after_uniprot['Gene Name'] = unmapped_proteins_after_uniprot['Gene Names'].dropna().apply(lambda s: re.split(';|,', s))
    unmapped_proteins_after_uniprot = unmapped_proteins_after_uniprot.explode('Gene Name')
    mapped_proteins_after_genename = unmapped_proteins_after_uniprot.reset_index(
        ).merge(protein_df[['PROTEIN_ID', 'GENE_NAME']],
                           left_on = 'Gene Name',
                           right_on = 'GENE_NAME',
                           how='left'
        ).drop(['Gene Name', 'GENE_NAME', 'Gene Names'], axis=1
        ).rename({'PROTEIN_ID':'PROTEIN_ID_FROM_GENE_NAME'},
                                        axis=1).dropna()
    #To resolve ambiguities, group by row index and retain only the smallest protein id 
    #(since canonical proteins were inserted first, this should be the most canonical isoform possible)
    unique_mapped_proteins_after_genename = mapped_proteins_after_genename.groupby('index').agg('min')
    input_csv_df = input_csv_df.merge(
        unique_mapped_proteins_after_genename,
        left_index=True,
        right_index=True,
        how='left')

In [28]:
if 'PROTEIN_ID_FROM_UNIPROT' in input_csv_df and 'PROTEIN_ID_FROM_GENE_NAME' in input_csv_df:
    input_csv_df['PROTEIN_ID'] = input_csv_df.apply(clean_protein_id, axis=1).astype('Int64')
    input_csv_df.drop(['PROTEIN_ID_FROM_UNIPROT', 'PROTEIN_ID_FROM_GENE_NAME'], axis=1, inplace=True)
elif 'PROTEIN_ID_FROM_UNIPROT' in input_csv_df:
    input_csv_df.rename({'PROTEIN_ID_FROM_UNIPROT': 'PROTEIN_ID'}, axis=1, inplace=True)
elif 'PROTEIN_ID_FROM_GENE_NAME' in input_csv_df:
    input_csv_df.rename({'PROTEIN_ID_FROM_GENE_NAME': 'PROTEIN_ID'}, axis=1, inplace=True)

In [32]:
input_csv_df_detail = input_csv_df[detail_columns].stack().reset_index()
input_csv_df_detail.set_index('level_0', inplace=True)
input_csv_df_detail.columns = ['KEY', 'VALUE']
input_csv_df_detail

,KEY,VALUE
level_0,,
0,Log Fold Change,-0.641359
0,Adjusted pvalue,0.646447
1,Log Fold Change,-2.232358
1,Adjusted pvalue,0.562901
2,Log Fold Change,-0.443683
...,...,...
2491,Adjusted pvalue,0.206656
2492,Log Fold Change,-1.132761
2492,Adjusted pvalue,0.629816


In [35]:
input_csv_df['DATASET_ID'] = dataset_id
if 'Experiment' not in input_csv_df:
    input_csv_df['Experiment'] = request['parameters']['datasetName']
input_csv_df

,Gene Names,Uniprot,Log Fold Change,Adjusted pvalue,Regulation,PROTEIN_ID,DATASET_ID,Experiment
0,HNRNPL,P14866,-0.641359,0.646447,None,11525.0,7,MyTestProteinDataset
1,EPS8L2,Q9H6S3,-2.232358,0.562901,None,14187.0,7,MyTestProteinDataset
2,USP11,P51784,-0.443683,0.591849,None,2123.0,7,MyTestProteinDataset
3,PTPN14,Q15678,-0.338962,0.372046,None,19845.0,7,MyTestProteinDataset
4,ATRX,P46100,-0.898034,0.722569,None,15935.0,7,MyTestProteinDataset
...,...,...,...,...,...,...,...,...
2489,LRCH4,O75427,-1.837674,0.665034,None,14116.0,7,MyTestProteinDataset
2490,TOMM22,Q9NS69,-0.769022,0.523625,None,9492.0,7,MyTestProteinDataset
2491,TRAF2,Q12933,2.494855,0.206656,None,12343.0,7,MyTestProteinDataset
2492,ZNF579,Q8NAF0,-1.132761,0.629816,None,1162.0,7,MyTestProteinDataset


In [36]:
with get_db_connection() as conn:
    next_user_datum_id = conn.execute('SELECT MAX(USER_DATUM_ID)+1 FROM USER_QUANTIFICATION_DATA').fetchall()[0][0]
    if not next_user_datum_id:
        next_user_datum_id = 1
next_user_datum_id

48649

In [40]:
input_csv_df.index+=next_user_datum_id
input_csv_df_detail.index+=next_user_datum_id

In [44]:
uqd_df = input_csv_df.reset_index().rename(
{
    'index': 'USER_DATUM_ID',
    'Experiment': 'EXPERIMENT',
    'Regulation': 'REGULATION',
}, axis=1)
uqd_df
#uqd_df[['USER_DATUM_ID','DATASET_ID','REGULATION','PROTEIN_ID','PEPTIDE_ID', 'MODIFIED_SEQUENCE','EXPERIMENT']]

,USER_DATUM_ID,Gene Names,Uniprot,Log Fold Change,Adjusted pvalue,REGULATION,PROTEIN_ID,DATASET_ID,EXPERIMENT
0,48649,HNRNPL,P14866,-0.641359,0.646447,None,11525.0,7,MyTestProteinDataset
1,48650,EPS8L2,Q9H6S3,-2.232358,0.562901,None,14187.0,7,MyTestProteinDataset
2,48651,USP11,P51784,-0.443683,0.591849,None,2123.0,7,MyTestProteinDataset
3,48652,PTPN14,Q15678,-0.338962,0.372046,None,19845.0,7,MyTestProteinDataset
4,48653,ATRX,P46100,-0.898034,0.722569,None,15935.0,7,MyTestProteinDataset
...,...,...,...,...,...,...,...,...,...
2489,51138,LRCH4,O75427,-1.837674,0.665034,None,14116.0,7,MyTestProteinDataset
2490,51139,TOMM22,Q9NS69,-0.769022,0.523625,None,9492.0,7,MyTestProteinDataset
2491,51140,TRAF2,Q12933,2.494855,0.206656,None,12343.0,7,MyTestProteinDataset
2492,51141,ZNF579,Q8NAF0,-1.132761,0.629816,None,1162.0,7,MyTestProteinDataset


In [ ]:
with get_db_connection() as conn:
    uqd_df[
        ['USER_DATUM_ID','DATASET_ID','REGULATION','PROTEIN_ID','EXPERIMENT']
        ].to_sql(
        'USER_QUANTIFICATION_DATA',
        conn,
        if_exists='append',
        index=False)
    input_csv_df_detail.reset_index(names='USER_DATUM_ID').to_sql(
        'USER_DATUM_DETAIL',
        conn,
        if_exists='append',
        index=False)

In [48]:
input_csv_df_detail

,KEY,VALUE
level_0,,
48649,Log Fold Change,-0.641359
48649,Adjusted pvalue,0.646447
48650,Log Fold Change,-2.232358
48650,Adjusted pvalue,0.562901
48651,Log Fold Change,-0.443683
...,...,...
51140,Adjusted pvalue,0.206656
51141,Log Fold Change,-1.132761
51141,Adjusted pvalue,0.629816


In [52]:
uqd_df

,USER_DATUM_ID,Gene Names,Uniprot,Log Fold Change,Adjusted pvalue,REGULATION,PROTEIN_ID,DATASET_ID,EXPERIMENT
0,48649,HNRNPL,P14866,-0.641359,0.646447,None,11525.0,7,MyTestProteinDataset
1,48650,EPS8L2,Q9H6S3,-2.232358,0.562901,None,14187.0,7,MyTestProteinDataset
2,48651,USP11,P51784,-0.443683,0.591849,None,2123.0,7,MyTestProteinDataset
3,48652,PTPN14,Q15678,-0.338962,0.372046,None,19845.0,7,MyTestProteinDataset
4,48653,ATRX,P46100,-0.898034,0.722569,None,15935.0,7,MyTestProteinDataset
...,...,...,...,...,...,...,...,...,...
2489,51138,LRCH4,O75427,-1.837674,0.665034,None,14116.0,7,MyTestProteinDataset
2490,51139,TOMM22,Q9NS69,-0.769022,0.523625,None,9492.0,7,MyTestProteinDataset
2491,51140,TRAF2,Q12933,2.494855,0.206656,None,12343.0,7,MyTestProteinDataset
2492,51141,ZNF579,Q8NAF0,-1.132761,0.629816,None,1162.0,7,MyTestProteinDataset
